# GEO Photometry (satellite tracking, integrated)
Written by Kiyoaki Okudaira<br>
*Kyushu University Hanada Lab / University of Washington / IAU CPS SatHub<br>
(okudaira.kiyoaki.528@s.kyushu-u.ac.jp or kiyoaki@uw.edu)<br>
<br>
Measure the light curve of a GEO object captured with the satellite-tracking method.<br>
Compared with `GEO_photometry_satTrack.ipynb`, this notebook<br>
- detects the target automatically (the object stays at one pixel position while field stars drift),<br>
- lets the operator review the detected position before the light curve is extracted (optional),<br>
- measures the photometric zeropoint from the field stars of the same capture (`magzero.ipynb` integrated).<br>
<br>
**Stages**<br>
1. Prepare frames (SER → FITS, calibration), build the detection image, list target candidates, save a review figure.<br>
2. Plate solve, zeropoint, field-star subtraction, aperture photometry, KUPT-lightcurveJSON output.<br>
<br>
**History**<br>
coding 2026-10-22 : 1st coding (based on GEO_photometry_satTrack 2026-07-07 and magzero 2026-07-05)<br>
<br>
(c) 2026 Kiyoaki Okudaira - Kyushu University Hanada Lab (SSDL) / University of Washington / IAU CPS SatHub

### Parameters
**Input files settings**

In [ ]:
import os

# Machine-specific roots (PATH_images, PATH_input, PATH_output, PATH_work)
# come from satphot_paths.py / paths.json.
# Intermediate FITS directories (fits_raw, fits_solved, ...) are written
# under PATH_work; products go to PATH_output.
from satphot_paths import *

# Input format: "SER" or "FITS"
input_format = "SER"

# For SER input, specify one SER file. For FITS input, specify a directory.
# (relative to PATH_images)
PATH_capture = PATH_images / "share2/KUPT/2026-10-20/SharpCap Captures/EKRAN 8/19_57_16.ser"

# Photometric zeropoint source
#   "integrated" : measure from the field stars of this capture (magzero integrated)
#   "file"       : use an existing MAGZERO JSON given by PATH_magzero
magzero_source = "integrated"
PATH_magzero = PATH_output / "magzero/magzero_2026-04-05T104510_GAIN200.json"

**Run control**

In [ ]:
# Which stage(s) to run in this session: "all", "1" or "2".
# Stage 2 reloads the stage-1 state saved under PATH_work when run alone.
run_stage = os.environ.get("SATPHOT_STAGE", "all")

# Ask the operator to confirm the detected target before stage 2.
# Set SATPHOT_REQUIRE_CONFIRMATION=0 (or require_confirmation = False) to run
# unattended.
require_confirmation = os.environ.get("SATPHOT_REQUIRE_CONFIRMATION", "1") != "0"

# Approval flag. Interactive use: set target_approved = True in the review
# cell. Headless use: SATPHOT_TARGET_APPROVED=1.
target_approved = os.environ.get("SATPHOT_TARGET_APPROVED", "0") == "1"

**Target and observation information**

In [ ]:
# Artificial-object information
object_name = "EKRAN 8"
intl_des    = "1982-009A"
norad_id    = "13056"

# Observer and observatory information
observer_name = "Kyushu University Hanada Lab (SSDL)"
observatory   = "KUPT Kyushu University Pegasus Telescope"

**SER files settings**

In [ ]:
# SER frame range (1-based, inclusive). Use None for the first/last frame.
ser_start_frame = None
ser_end_frame = None

# Camera settings used when converting SER to FITS.
# Values found in a SharpCap CameraSettings.txt file override these values.
ser_exposure_seconds = 1.0 / 8.0
ser_gain = None
ser_egain = None
ser_egainsav = None
ser_offset = None
ser_x_binning = 1
ser_y_binning = 1
ser_camera_name = None
ser_camera_id = None
ser_ccd_temperature = None
ser_set_temperature = None

**Calibration settings**

In [ ]:
# Calibration files (single FITS or a directory of FITS). Set to None when not used.
# Dark/flat are applied AFTER astrometry: plate solving uses the raw frames,
# the calibrated copies (PATH_work/fits_cal) are used for the zeropoint,
# field-star subtraction and photometry.
PATH_dark = PATH_capture.parent / "darks/MasterDark_20_01_36.fits"
PATH_flat = PATH_capture.parent / "flats/MasterFlat_20_03_53_offset=0.098%.fits"
PATH_flat_dark = None

# Scale applied to the master dark before subtraction.
# "auto": SharpCap stores 8-bit master frames as 16-bit values (x256); when the
# light frames are 8-bit and the master frame exceeds 255, 1/256 is used.
# Give a number to force a scale (1.0 = none).
calibration_scale = "auto"

**Plate solve settings**

In [ ]:
# Astrometry mode
#   "stacks": consecutive frames are aligned on the drifting stars and summed,
#             the stacks are plate-solved, and every frame receives a WCS from a
#             linear drift model plus the per-frame tracking jitter measured on
#             the target. Needed when single frames show too few stars
#             (e.g. 8-bit short exposures).
#   "frames": every frame is plate-solved individually (GEO_photometry_satTrack).
astrometry_mode = "stacks"
astrometry_stack_frames = 31     # frames summed per stack
astrometry_stack_count = 19      # stacks spread over the capture (the WCS of each frame is
                                 # interpolated between them, so more stacks follow tracking-rate changes better)

# Star drift per frame (dx, dy) [pixel]. None = estimate automatically from the frames.
star_drift_px_per_frame = None

# Astrometry.net settings. The pixel-scale range is important with few field
# stars: KUPT QHY268M (ROI) is 0.203 arcsec/pixel, KUBO QHY461M 0.227.
astrometry_downsample = 2
astrometry_search_radius_deg = 2.0
astrometry_scale_low_arcsec_per_pixel = 0.18
astrometry_scale_high_arcsec_per_pixel = 0.22
astrometry_cpulimit_seconds = 600

# Sky position hint (RA, Dec) [deg].
# "auto" reads the telescope position (SiTechDll Telescope=RA=..,Dec=..) from
# the SharpCap CameraSettings file next to the SER file, falling back to
# LastPlateSolveData; a tuple gives it explicitly; None disables the hint.
astrometry_center_hint = "auto"

# Star list sent to solve-field (stack mode): sources are extracted here with
# an elongation filter so that noise and the target streak are excluded.
astrometry_star_nsigma = 5.0
astrometry_star_min_area = 8
astrometry_star_max_elongation = 2.0
astrometry_star_max_count = 40
astrometry_extra_options = {
    "--objs": 50,
    "--crpix-center": True,
    "--no-remove-lines": True,
    "--uniformize": 0,
    "--overwrite": True,
    "--no-plots": True,
    "--fits-image": True,   # frame mode only
    "--tag-all": True,      # keep catalogue magnitudes in the RDLS file (needed for the zeropoint)
}
astrometry_max_workers = 16

**Star subtraction settings**

In [ ]:
# Star subtraction settings
subtract_stars = True
star_reference_interval = 16
star_shift_order = 3
star_shift_mode = "constant"
star_shift_cval = float("nan")

# Run field-star subtraction in parallel
star_subtraction_parallel = True
star_subtraction_max_workers = 16

**Target detection settings**<br>In satellite-tracking data the target stays at one pixel position while the stars drift, so the median of frames sampled across the capture keeps the target and rejects the stars.

In [ ]:
# Number of frames (evenly spaced over the capture) combined into the detection image
detection_sample_count = 31

# Detection threshold on the detection image [sigma above background]
detection_sigma = 6.0

# Gaussian smoothing applied before thresholding [pixel]
detection_smoothing_sigma = 1.0

# Minimum connected area of a candidate [pixel] (rejects hot pixels)
detection_min_area = 9

# A candidate whose brightest pixel holds more than this fraction of its flux
# is a hot pixel / cosmic ray, not a PSF; it is dropped before ranking.
detection_max_peak_fraction = 0.15

# Candidates more elongated than this (sqrt of the second-moment axis ratio)
# are demoted; residual star streaks are elongated, the target is round.
detection_max_elongation = 2.5

# Aperture radius used for the per-frame persistence check [pixel]
detection_snr_radius = 5.0

# Pixel value regarded as saturated (8-bit SER: 255)
saturation_level = 250

# Number of candidates listed in the review figure
n_candidates = 5

# Hot-pixel removal. Warm pixels of the master dark (when given) are masked
# before the target detection; pixels that stay bright in the detection image
# (outside the candidate neighbourhood) are added afterwards. Masked pixels are
# replaced by the local median whenever a frame is read.
remove_hot_pixels = True
hot_pixel_sigma = 5.0
hot_pixel_exclude_radius = 40

# Candidate adopted as the target (0 = best ranked). Change after the review
# if another candidate is the object.
target_candidate_index = 0

# Manual override: set both to numbers to ignore the automatic detection.
target_x = None
target_y = None

**Measurement settings**

In [ ]:
# Keep the same target position in every frame. Set True only when the mount
# tracking is steady; with tracking jitter the target is re-centred in every
# frame around the detected position (recommended).
fix_target_position = False

# Per-frame re-centring: search window half-size around the detected position
# and the largest accepted offset from it [pixel]. A frame whose centroid lies
# farther away falls back to the detected position.
centroid_search_half_size = 25
centroid_max_shift = 30

# Centroid refinement of the candidates on the detection image [pixel]
centroid_half_size = 8

# Aperture photometry settings [pixel]
aperture_radius    = 12.0
annulus_r_in       = 18.0
annulus_r_out      = 24.0

**Zeropoint settings** (used when magzero_source = "integrated")

In [ ]:
# Number of plate-solved frames (evenly spaced) on which the zeropoint is measured
zeropoint_frame_count = 5

# Aperture for catalogue stars [pixel]. Stars drift during the exposure, so the
# aperture must cover the streak (sidereal rate x exposure / pixel scale).
zp_aperture_radius = 16
zp_annulus_r_in    = 22
zp_annulus_r_out   = 28

# Star centroid refinement
zp_box_size  = 41    # [pixel] MUST BE ODD
zp_max_shift = 12    # [pixel]

# Catalogue magnitude range of reference stars
reference_mag_min = 8.0
reference_mag_max = 14.0

**Software metadata**<br>Please DO NOT change from original.

In [ ]:
# Output software metadata
editor_app_name  = "SatPhotometry GEO SatTrack integrated"
editor_version = "1.0.0"
creator_app_name = "SatPhotometry GEO SatTrack integrated"
creator_version  = "1.0.0"

### Import and initial settings

In [ ]:
import json
import platform
import re
import shutil
import warnings
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.stats import SigmaClip, sigma_clipped_stats
from astropy.table import Table
from astropy.time import Time, TimeDelta
from astropy.visualization import ImageNormalize, ZScaleInterval, AsinhStretch
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.wcs.wcs import FITSFixedWarning
from photutils.aperture import (
    ApertureStats,
    CircularAnnulus,
    CircularAperture,
    aperture_photometry,
)
from photutils.centroids import centroid_2dg, centroid_sources
from tqdm.auto import tqdm
from IPython.display import Image, display

from concurrent.futures import ThreadPoolExecutor, as_completed

from satphotometry import astrometry
from satphotometry import serparser

warnings.simplefilter("ignore", FITSFixedWarning)

### General utilities

In [ ]:
def get_fits_files(directory):
    """Return non-hidden FITS files directly contained in a directory."""
    directory = Path(directory)
    if not directory.is_dir():
        raise NotADirectoryError(f"FITS directory does not exist: {directory}")

    files = [
        p for p in directory.iterdir()
        if p.is_file()
        and not p.name.startswith(".")
        and p.suffix.lower() in {".fit", ".fits", ".fts"}
    ]
    files.sort(key=lambda p: p.name.casefold())
    if not files:
        raise FileNotFoundError(f"No FITS files were found in: {directory}")
    return files


def recreate_directory(directory):
    """Delete and recreate a processing directory."""
    directory = Path(directory)
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)
    return directory


def get_observation_time(header):
    """Return exposure start, midpoint, and duration from a FITS header."""
    date_obs = header.get("DATE-OBS", header.get("DATEOBS"))
    if date_obs is None:
        raise KeyError("The FITS header contains neither DATE-OBS nor DATEOBS.")
    if "EXPTIME" not in header:
        raise KeyError("The FITS header does not contain EXPTIME.")

    exptime = float(header["EXPTIME"])
    if not np.isfinite(exptime) or exptime <= 0:
        raise ValueError(f"Invalid EXPTIME: {exptime}")

    try:
        time_start = Time(str(date_obs).strip(), format="fits", scale="utc")
    except ValueError:
        time_start = Time(str(date_obs).strip(), scale="utc")

    time_mid = time_start + TimeDelta(exptime / 2.0, format="sec")
    return time_start, time_mid, exptime


def windows_ticks_to_time(value):
    """Convert SER/Windows ticks since 0001-01-01 to Astropy Time."""
    unix_seconds = float(value) / 10_000_000.0 - 62_135_596_800.0
    return Time(unix_seconds, format="unix", scale="utc")


def parse_number(text):
    """Return the first floating-point number contained in text."""
    if text is None:
        return None
    match = re.search(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", str(text))
    return None if match is None else float(match.group(0))


def parse_sharpcap_settings(path):
    """Read commonly used fields from a SharpCap CameraSettings.txt file."""
    path = Path(path)
    if not path.exists():
        return {}

    settings = {}
    with path.open("r", encoding="utf-8", errors="replace") as file:
        for line in file:
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            settings[key.strip()] = value.strip().strip('"')

    result = {}
    result["camera_name"] = settings.get("Camera") or settings.get("CameraName")
    result["camera_id"] = settings.get("CameraSerialNumber")

    binning = settings.get("Binning")
    if binning and "x" in binning.lower():
        x_bin, y_bin = re.split("x", binning, maxsplit=1, flags=re.IGNORECASE)
        result["x_binning"] = int(parse_number(x_bin))
        result["y_binning"] = int(parse_number(y_bin))

    exposure_text = settings.get("Exposure")
    exposure_value = parse_number(exposure_text)
    if exposure_value is not None:
        if "ms" in exposure_text.lower():
            exposure_value /= 1000.0
        elif "us" in exposure_text.lower() or "µs" in exposure_text.lower():
            exposure_value /= 1_000_000.0
        result["exposure"] = exposure_value

    key_map = {
        "Gain": "gain",
        "Offset": "offset",
        "Temperature": "ccd_temperature",
        "Target Temperature": "set_temperature",
    }
    for source_key, destination_key in key_map.items():
        value = parse_number(settings.get(source_key))
        if value is not None:
            result[destination_key] = value

    if "SharpCapVersion" in settings:
        result["swcreate"] = f"SharpCap {settings['SharpCapVersion']}"
    return result

### SER conversion and calibration

In [ ]:
def convert_ser_to_fits(
    ser_path,
    output_directory,
    *,
    start_frame=None,
    end_frame=None,
    exposure_seconds,
    gain=None,
    egain=None,
    egainsav=None,
    offset=None,
    x_binning=1,
    y_binning=1,
    camera_name=None,
    camera_id=None,
    ccd_temperature=None,
    set_temperature=None,
    observer=None,
    observatory=None,
    object_name=None,
    intl_des=None,
    norad_id=None,
):
    """Convert selected SER frames to FITS using satphotometry.serparser."""
    ser_path = Path(ser_path)
    output_directory = recreate_directory(output_directory)
    ser_file = serparser.Serfile(str(ser_path))
    frame_count = int(ser_file.getLength())

    first = 1 if start_frame is None else int(start_frame)
    last = frame_count if end_frame is None else int(end_frame)
    if first < 1 or last > frame_count or first > last:
        raise ValueError(
            f"Invalid SER frame range: {first}--{last}; total={frame_count}"
        )

    camera_settings_path = ser_path.with_suffix(".CameraSettings.txt")
    camera_settings = parse_sharpcap_settings(camera_settings_path)

    exposure_seconds = camera_settings.get("exposure", exposure_seconds)
    gain = camera_settings.get("gain", gain)
    offset = camera_settings.get("offset", offset)
    x_binning = camera_settings.get("x_binning", x_binning)
    y_binning = camera_settings.get("y_binning", y_binning)
    camera_name = camera_settings.get("camera_name", camera_name)
    camera_id = camera_settings.get("camera_id", camera_id)
    ccd_temperature = camera_settings.get("ccd_temperature", ccd_temperature)
    set_temperature = camera_settings.get("set_temperature", set_temperature)
    swcreate = camera_settings.get("swcreate", "satphotometry.serparser")

    try:
        timestamps = ser_file.readTrailFromHeader()
        if len(timestamps) != frame_count:
            timestamps = None
    except Exception:
        timestamps = None

    if timestamps is None:
        raise RuntimeError(
            "The SER file does not contain a usable per-frame timestamp trail. "
            "Reliable light-curve timing requires SER timestamps."
        )

    output_files = []
    stem = ser_path.stem
    for zero_index in tqdm(range(first - 1, last), desc="Converting SER to FITS"):
        output_path = output_directory / f"{stem}_{zero_index + 1:05d}.fits"
        ser_file.setCurrentPosition(zero_index)
        ser_file.saveFit(str(output_path))

        with fits.open(output_path, mode="update", memmap=False) as hdul:
            header = hdul[0].header
            time_start = windows_ticks_to_time(timestamps[zero_index])
            header["DATE-OBS"] = (time_start.fits, "Exposure start time UTC")
            header["EXPTIME"] = (float(exposure_seconds), "Exposure time [s]")
            header["XBINNING"] = int(x_binning)
            header["YBINNING"] = int(y_binning)
            header["SWCREATE"] = swcreate

            optional_header = {
                "GAIN": gain,
                "EGAIN": egain,
                "EGAINSAV": egainsav,
                "OFFSET": offset,
                "INSTRUME": camera_name,
                "CAMID": camera_id,
                "CCD-TEMP": ccd_temperature,
                "SET-TEMP": set_temperature,
                "OBSERVER": observer,
                "OBSERVAT": observatory,
                "OBJECT": object_name,
                "INTL-DES": intl_des,
                "NORAD-ID": norad_id,
            }
            for key, value in optional_header.items():
                if value is not None:
                    header[key] = value
            hdul.flush(output_verify="silentfix")

        output_files.append(output_path)
    return output_files


def create_master_dark(path):
    """Read a single dark frame or median-combine dark frames in a directory."""
    if path is None:
        return None
    path = Path(path)
    files = get_fits_files(path) if path.is_dir() else [path]
    stack = []
    for file_path in files:
        with fits.open(file_path, memmap=False) as hdul:
            stack.append(np.asarray(hdul[0].data, dtype=float))
    return np.nanmedian(np.stack(stack), axis=0)


def create_master_flat(path, master_flat_dark=None):
    """Read and normalize a single flat or a median-combined flat stack."""
    if path is None:
        return None
    path = Path(path)
    files = get_fits_files(path) if path.is_dir() else [path]
    stack = []
    for file_path in files:
        with fits.open(file_path, memmap=False) as hdul:
            data = np.asarray(hdul[0].data, dtype=float)
        if master_flat_dark is not None:
            data = data - master_flat_dark
        stack.append(data)
    master = np.nanmedian(np.stack(stack), axis=0)
    norm = np.nanmedian(master[np.isfinite(master)])
    if not np.isfinite(norm) or norm == 0:
        raise ValueError("The master flat cannot be normalized.")
    return master / norm


def calibrate_fits_files(fits_files, dark=None, flat=None):
    """Apply dark subtraction and flat-field correction in place."""
    if dark is None and flat is None:
        return
    for file_path in tqdm(fits_files, desc="Calibrating FITS frames"):
        with fits.open(file_path, mode="update", memmap=False) as hdul:
            data = np.asarray(hdul[0].data, dtype=float)
            if dark is not None:
                data = data - dark
            if flat is not None:
                with np.errstate(divide="ignore", invalid="ignore"):
                    data = data / flat
            hdul[0].data = data.astype(np.float32)
            hdul.flush(output_verify="silentfix")

### Plate solving

In [ ]:
def build_astrometry_options(output_path, work_directory, previous_radec=None):
    """Build solve-field options accepted by satphotometry.astrometry.platesolve."""
    work_directory = Path(work_directory)
    work_directory.mkdir(parents=True, exist_ok=True)
    options = {
        "-N": str(output_path),
        "--dir": str(work_directory),
        "--downsample": int(astrometry_downsample),
    }
    options.update(astrometry_extra_options)

    if astrometry_scale_low_arcsec_per_pixel is not None:
        options["--scale-low"] = astrometry_scale_low_arcsec_per_pixel
        options["--scale-units"] = "arcsecperpix"
    if astrometry_scale_high_arcsec_per_pixel is not None:
        options["--scale-high"] = astrometry_scale_high_arcsec_per_pixel
        options["--scale-units"] = "arcsecperpix"

    if previous_radec is not None:
        options["--ra"] = float(previous_radec[0])
        options["--dec"] = float(previous_radec[1])
        options["--radius"] = float(astrometry_search_radius_deg)
    return options

def plate_solve_single_frame(task):
    """
    Plate-solve one frame in a separate process.
    """

    (
        frame_index,
        file_path,
        solved_path,
        frame_work,
        reference_radec,
    ) = task

    file_path = Path(file_path)
    solved_path = Path(solved_path)
    frame_work = Path(frame_work)

    frame_work.mkdir(
        parents=True,
        exist_ok=True,
    )

    options = build_astrometry_options(
        output_path=solved_path,
        work_directory=frame_work,
        previous_radec=reference_radec,
    )

    try:
        result_path, solved = astrometry.platesolve(
            str(file_path),
            options,
            async_process=False,
            jupyter_env=True,
        )

        if not solved:
            return {
                "frame_index": frame_index,
                "source_path": file_path,
                "solved_path": Path(result_path),
                "solved": False,
                "error": "solve-field did not solve the frame.",
            }

        result_path = Path(result_path)

        if not result_path.exists():
            return {
                "frame_index": frame_index,
                "source_path": file_path,
                "solved_path": result_path,
                "solved": False,
                "error": "The solved FITS file was not created.",
            }

        return {
            "frame_index": frame_index,
            "source_path": file_path,
            "solved_path": result_path,
            "solved": True,
            "error": None,
        }

    except Exception as error:
        return {
            "frame_index": frame_index,
            "source_path": file_path,
            "solved_path": solved_path,
            "solved": False,
            "error": str(error),
        }

def solve_reference_frame(
    file_path,
    solved_directory,
    work_directory,
):
    """
    Solve one reference frame without an RA/Dec constraint.
    """

    file_path = Path(file_path)

    solved_path = (
        Path(solved_directory)
        / file_path.name
    )

    frame_work = (
        Path(work_directory)
        / f"{file_path.stem}_reference"
    )

    options = build_astrometry_options(
        output_path=solved_path,
        work_directory=frame_work,
        previous_radec=None,
    )

    result_path, solved = astrometry.platesolve(
        str(file_path),
        options,
        async_process=True,
        jupyter_env=True,
    )

    if not solved or not Path(result_path).exists():
        raise RuntimeError(
            "The reference frame could not be plate-solved."
        )

    with fits.open(
        result_path,
        memmap=False,
    ) as hdul:
        data_shape = hdul[0].data.shape
        solved_header = hdul[0].header.copy()

    reference_wcs = WCS(
        solved_header
    )

    center_x = (
        data_shape[1] - 1
    ) / 2.0

    center_y = (
        data_shape[0] - 1
    ) / 2.0

    center_sky = reference_wcs.pixel_to_world(
        center_x,
        center_y,
    )

    reference_radec = (
        float(center_sky.ra.deg),
        float(center_sky.dec.deg),
    )

    return (
        Path(result_path),
        reference_radec,
    )

def plate_solve_files_parallel(
    fits_files,
    solved_directory,
    work_directory,
    max_workers=4,
):
    """
    Plate-solve frames in parallel using one common
    RA/Dec search center.
    """

    solved_directory = recreate_directory(
        solved_directory
    )

    work_directory = recreate_directory(
        work_directory
    )

    fits_files = [
        Path(file_path)
        for file_path in fits_files
    ]

    if len(fits_files) == 0:
        raise ValueError(
            "No FITS files were supplied."
        )

    # Use the middle frame as the reference because it is less
    # likely to lie at the edge of the total star-motion range.
    reference_index = (
        len(fits_files) // 2
    )

    reference_file = fits_files[
        reference_index
    ]

    (
        reference_solved_path,
        reference_radec,
    ) = solve_reference_frame(
        file_path=reference_file,
        solved_directory=solved_directory,
        work_directory=work_directory,
    )

    print(
        "Reference astrometry center: "
        f"RA={reference_radec[0]:.6f} deg, "
        f"Dec={reference_radec[1]:.6f} deg"
    )

    tasks = []

    for frame_index, file_path in enumerate(
        fits_files
    ):
        solved_path = (
            solved_directory
            / file_path.name
        )

        frame_work = (
            work_directory
            / file_path.stem
        )

        tasks.append(
            (
                frame_index,
                str(file_path),
                str(solved_path),
                str(frame_work),
                reference_radec,
            )
        )

    raw_results = []

    with ThreadPoolExecutor(
        max_workers=max_workers,
    ) as executor:
        future_to_index = {
            executor.submit(
                plate_solve_single_frame,
                task,
            ): task[0]
            for task in tasks
        }

        for future in tqdm(
            as_completed(future_to_index),
            total=len(future_to_index),
            desc="Plate solving",
        ):
            frame_index = future_to_index[
                future
            ]

            try:
                result = future.result()
            except Exception as error:
                result = {
                    "frame_index": frame_index,
                    "source_path": (
                        fits_files[frame_index]
                    ),
                    "solved_path": (
                        solved_directory
                        / fits_files[frame_index].name
                    ),
                    "solved": False,
                    "error": str(error),
                }

            raw_results.append(
                result
            )

    # Restore chronological frame order.
    raw_results.sort(
        key=lambda item: item[
            "frame_index"
        ]
    )

    records = []

    for result in raw_results:
        wcs = None
        header = None

        if (
            result["solved"]
            and result["solved_path"].exists()
        ):
            try:
                with fits.open(
                    result["solved_path"],
                    memmap=False,
                ) as hdul:
                    header = (
                        hdul[0].header.copy()
                    )

                wcs = WCS(
                    header
                )

            except Exception as error:
                result["solved"] = False
                result["error"] = (
                    "The solved FITS file could not be read: "
                    f"{error}"
                )

        if result["error"] is not None:
            print(
                "Plate solve failed for "
                f"{result['source_path'].name}: "
                f"{result['error']}"
            )

        records.append(
            {
                "source_path": (
                    result["source_path"]
                ),
                "solved_path": (
                    result["solved_path"]
                ),
                "solved": bool(
                    wcs is not None
                ),
                "wcs": wcs,
                "header": header,
            }
        )

    return records

def interpolate_star_positions(records):
    """Interpolate a common sky coordinate's pixel position in failed frames."""
    valid_indices = [i for i, record in enumerate(records) if record["wcs"] is not None]
    if len(valid_indices) < 2:
        raise RuntimeError("At least two successfully plate-solved frames are required.")

    reference_index = valid_indices[0]
    reference_record = records[reference_index]
    with fits.open(reference_record["source_path"], memmap=False) as hdul:
        ny, nx = np.asarray(hdul[0].data).shape
    center_x = (nx - 1) / 2.0
    center_y = (ny - 1) / 2.0
    common_sky = reference_record["wcs"].pixel_to_world(center_x, center_y)

    x_values = np.full(len(records), np.nan, dtype=float)
    y_values = np.full(len(records), np.nan, dtype=float)
    for i, record in enumerate(records):
        if record["wcs"] is not None:
            x_values[i], y_values[i] = record["wcs"].world_to_pixel(common_sky)

    indices = np.arange(len(records), dtype=float)
    x_values = np.interp(indices, valid_indices, x_values[valid_indices])
    y_values = np.interp(indices, valid_indices, y_values[valid_indices])
    return x_values, y_values

### Astrometry from star-aligned stacks (satellite tracking)

In [ ]:
def sanitize_json(value):
    '''Recursively convert NaN/inf and NumPy scalars for json.dump(allow_nan=False).'''
    if isinstance(value, dict):
        return {str(k): sanitize_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [sanitize_json(v) for v in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def read_center_hint():
    '''(RA, Dec) [deg] from SharpCap's CameraSettings next to the SER file, or None.'''
    if astrometry_center_hint is None:
        return None
    if astrometry_center_hint != "auto":
        return float(astrometry_center_hint[0]), float(astrometry_center_hint[1])
    path = PATH_capture.with_suffix(".CameraSettings.txt")
    if not path.is_file():
        return None
    text = path.read_text(encoding="utf-8", errors="replace")
    pattern = r"RA=(\d+):(\d+):([\d.]+),\s*Dec=([+-]?\d+):(\d+):([\d.]+)"
    telescope = re.search(r"Telescope=" + pattern, text)
    match = telescope or re.search(pattern, text)
    if match is None:
        return None
    h, m, sec, dd, dm, ds = match.groups()
    ra = 15.0 * (float(h) + float(m) / 60.0 + float(sec) / 3600.0)
    sign = -1.0 if dd.strip().startswith("-") else 1.0
    dec = sign * (abs(float(dd)) + float(dm) / 60.0 + float(ds) / 3600.0)
    return ra, dec


def build_dark_hot_pixel_mask(master_dark, sigma):
    '''Warm pixels of the master dark (robust sigma clipping with a MAD scale).'''
    dark = np.asarray(master_dark, dtype=float)
    finite = np.isfinite(dark)
    median = np.median(dark[finite])
    mad = np.median(np.abs(dark[finite] - median))
    scale = max(1.4826 * mad, estimate_noise(dark[finite], floor=0.05))
    return finite & (dark > median + sigma * scale)


def build_hot_pixel_mask(detection_image, candidates, sigma, exclude_radius):
    '''Static bright pixels in the detection image, excluding the candidate neighbourhoods.'''
    image = np.asarray(detection_image, dtype=float)
    finite = np.isfinite(image)
    _, median, _ = sigma_clipped_stats(image[finite], sigma=3.0, maxiters=5)
    std = estimate_noise(image[finite])
    mask = finite & (image > median + sigma * std)
    yy, xx = np.indices(image.shape)
    for c in candidates:
        mask &= np.hypot(xx - c["x"], yy - c["y"]) > exclude_radius
    return mask


def calibrate_frames(fits_files, output_directory, dark=None, flat=None, max_workers=16):
    '''
    Write dark/flat-corrected, hot-pixel-cleaned float32 copies of the raw
    frames (used for the zeropoint, star subtraction and photometry; the raw
    frames stay untouched for astrometry).
    '''
    output_directory = recreate_directory(output_directory)

    def calibrate_one(path):
        with fits.open(path, memmap=False) as hdul:
            data = np.asarray(hdul[0].data, dtype=np.float32)
            header = hdul[0].header.copy()
        if dark is not None:
            data = data - dark
        if flat is not None:
            with np.errstate(divide="ignore", invalid="ignore"):
                data = data / flat
        data = clean_hot(data.astype(np.float32))
        header["CALIB"] = (True, "Dark/flat applied; hot pixels replaced")
        header["DARKSUB"] = (dark is not None, "Master dark subtracted")
        header["FLATDIV"] = (flat is not None, "Divided by normalised master flat")
        output_path = output_directory / path.name
        fits.writeto(output_path, data, header, overwrite=True, output_verify="silentfix")
        return output_path

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(tqdm(executor.map(calibrate_one, fits_files), total=len(fits_files), desc="Calibrating frames"))


def find_blobs(image, nsigma=5.0, min_area=6, n_max=30, smoothing=1.5):
    '''Brightest compact sources of one frame (x, y, peak, area).'''
    smoothed = ndimage.gaussian_filter(np.nan_to_num(image, nan=0.0), smoothing)
    _, median, _ = sigma_clipped_stats(smoothed[::4, ::4], sigma=3.0, maxiters=5)
    std = estimate_noise(smoothed[::4, ::4], floor=0.05)
    labels, number = ndimage.label(smoothed > median + nsigma * std)
    blobs = []
    for k, slices in enumerate(ndimage.find_objects(labels), start=1):
        region = labels[slices] == k
        if np.count_nonzero(region) < min_area:
            continue
        weights = np.where(region, smoothed[slices] - median, 0.0)
        total = weights.sum()
        yy, xx = np.indices(weights.shape)
        xc = (weights * xx).sum() / total
        yc = (weights * yy).sum() / total
        cxx = (weights * (xx - xc) ** 2).sum() / total
        cyy = (weights * (yy - yc) ** 2).sum() / total
        cxy = (weights * (xx - xc) * (yy - yc)).sum() / total
        eigen = np.linalg.eigvalsh([[cxx, cxy], [cxy, cyy]])
        elongation = float(np.sqrt(eigen[1] / max(eigen[0], 1e-6)))
        blobs.append((
            float(xc) + slices[1].start,
            float(yc) + slices[0].start,
            float(total),
            int(np.count_nonzero(region)),
            elongation,
        ))
    blobs.sort(key=lambda b: -b[2])
    return blobs[:n_max]


def extract_star_list(image):
    '''Round, well-detected sources of a stack as an astrometry.net xylist table.'''
    ny, nx = image.shape
    blobs = find_blobs(image, nsigma=astrometry_star_nsigma, min_area=astrometry_star_min_area,
                       n_max=astrometry_star_max_count * 3)
    rows = [
        (x, y, flux) for x, y, flux, area, elongation in blobs
        if elongation <= astrometry_star_max_elongation and 5 < x < nx - 5 and 5 < y < ny - 5
    ]
    return Table(rows=rows[:astrometry_star_max_count], names=("X", "Y", "FLUX"))


def solve_star_list(xyls_path, width, height, work_directory, center_hint):
    '''Run solve-field on an xylist; return (wcs_header, rdls_path, error).'''
    work_directory = Path(work_directory)
    work_directory.mkdir(parents=True, exist_ok=True)
    stem = Path(xyls_path).stem
    command = [
        "solve-field", str(xyls_path), "--width", str(width), "--height", str(height),
        "--crpix-center", "--no-plots", "--overwrite", "--tag-all",
        "-D", str(work_directory), "--cpulimit", str(astrometry_cpulimit_seconds),
    ]
    for key, value in astrometry_extra_options.items():
        if key in {"--fits-image", "--downsample", "-z", "--no-plots", "--overwrite", "--tag-all", "--crpix-center"}:
            continue
        if value is True:
            command.append(key)
        elif value not in (False, None):
            command += [key, str(value)]
    if astrometry_scale_low_arcsec_per_pixel is not None:
        command += ["--scale-units", "arcsecperpix", "--scale-low", str(astrometry_scale_low_arcsec_per_pixel)]
    if astrometry_scale_high_arcsec_per_pixel is not None:
        command += ["--scale-units", "arcsecperpix", "--scale-high", str(astrometry_scale_high_arcsec_per_pixel)]
    if center_hint is not None:
        command += ["--ra", str(center_hint[0]), "--dec", str(center_hint[1]),
                    "--radius", str(astrometry_search_radius_deg)]
    import subprocess
    completed = subprocess.run(command, capture_output=True, text=True)
    wcs_path = work_directory / f"{stem}.wcs"
    rdls_path = work_directory / f"{stem}.rdls"
    if not (work_directory / f"{stem}.solved").exists() or not wcs_path.exists():
        tail = "\n".join(completed.stdout.strip().splitlines()[-3:])
        return None, None, f"solve-field did not solve the star list. {tail}"
    with fits.open(wcs_path) as hdul:
        wcs_header = hdul[0].header.copy()
    return wcs_header, rdls_path, None


def merge_wcs_header(header, wcs_header):
    '''Copy the WCS cards of a solve-field .wcs header into an image header.'''
    skip = {"SIMPLE", "BITPIX", "NAXIS", "NAXIS1", "NAXIS2", "EXTEND", "COMMENT", "HISTORY", "", "DATE"}
    merged = header.copy()
    for card in wcs_header.cards:
        if card.keyword in skip:
            continue
        merged[card.keyword] = (card.value, card.comment)
    return merged


def estimate_star_drift(fits_files, sat_x, sat_y, gaps=(20, 40), n_pairs=4, bin_px=0.25):
    '''
    Star drift per frame from the most common blob displacement between frame
    pairs, after removing the tracking jitter measured on the target.
    '''
    count = len(fits_files)
    votes = {}
    contributions = {}
    starts = np.linspace(0, count - 1 - max(gaps), n_pairs).round().astype(int)
    for gap in gaps:
        for a in starts:
            b = a + gap
            blobs_a = find_blobs(read_image(fits_files[a]))
            blobs_b = find_blobs(read_image(fits_files[b]))
            jitter = (sat_x[b] - sat_x[a], sat_y[b] - sat_y[a])
            for xa, ya, *_ in blobs_a:
                for xb, yb, *_ in blobs_b:
                    vx = (xb - xa - jitter[0]) / gap
                    vy = (yb - ya - jitter[1]) / gap
                    if np.hypot(vx, vy) < 0.3 or np.hypot(vx, vy) > 60:
                        continue
                    key = (round(vx / bin_px), round(vy / bin_px))
                    votes[key] = votes.get(key, 0) + 1
                    contributions.setdefault(key, []).append((vx, vy))
    if not votes:
        raise RuntimeError("No star displacement could be measured; set star_drift_px_per_frame manually.")
    best = max(votes, key=votes.get)
    neighbours = [k for k in votes if abs(k[0] - best[0]) <= 1 and abs(k[1] - best[1]) <= 1]
    samples = np.array([v for k in neighbours for v in contributions[k]])
    drift = np.median(samples, axis=0)
    n_votes = sum(votes[k] for k in neighbours)
    print(f"Star drift: ({drift[0]:.3f}, {drift[1]:.3f}) px/frame from {n_votes} matched blob pairs "
          f"(|v| = {np.hypot(*drift):.3f} px/frame)")
    if n_votes < 3:
        print("Warning: only a few matched pairs; check the drift or set star_drift_px_per_frame.")
    return float(drift[0]), float(drift[1])


def measure_target_positions(fits_files, x0, y0):
    '''Target position in every frame (re-centred around the detected position).'''
    count = len(fits_files)
    xs = np.full(count, x0, dtype=float)
    ys = np.full(count, y0, dtype=float)
    methods = ["fixed_position"] * count

    def locate_one(j):
        return locate_target(read_image(fits_files[j]), x0, y0, centroid_search_half_size, centroid_max_shift)

    with ThreadPoolExecutor(max_workers=16) as executor:
        for j, (x, y, method) in enumerate(
            tqdm(executor.map(locate_one, range(count)), total=count, desc="Locating target")
        ):
            xs[j], ys[j], methods[j] = x, y, method
    n_fallback = sum(1 for m in methods if m != "local_centroid")
    # Outlier rejection: a centroid far from the running median (a star streak
    # passing through the search window, a saturated flash) is replaced by it.
    window = 15
    xs_med = ndimage.median_filter(xs, size=window, mode="nearest")
    ys_med = ndimage.median_filter(ys, size=window, mode="nearest")
    resid = np.hypot(xs - xs_med, ys - ys_med)
    limit = max(6.0, 4.0 * 1.4826 * np.median(resid))
    outlier = resid > limit
    for j in np.nonzero(outlier)[0]:
        xs[j], ys[j], methods[j] = xs_med[j], ys_med[j], "track_smoothed"
    print(f"Target located in {count - n_fallback} / {count} frames "
          f"(mean x = {xs.mean():.2f}, y = {ys.mean():.2f}, rms jitter = "
          f"{np.hypot(xs - xs.mean(), ys - ys.mean()).std():.2f} px); "
          f"{int(outlier.sum())} outlier(s) beyond {limit:.1f} px replaced by the running median")
    return xs, ys, methods


def build_star_aligned_stacks(fits_files, sat_x, sat_y, drift, stack_frames, stack_count, output_directory):
    '''Sum frames aligned on the stars: shift(j) = (j - c) * drift + jitter(j).'''
    output_directory = recreate_directory(output_directory)
    count = len(fits_files)
    half = int(stack_frames) // 2
    centers = np.unique(np.linspace(half, count - 1 - half, int(stack_count)).round().astype(int))

    def build_one(center):
        members = range(center - half, center + half + 1)
        total = None
        weight = None
        for j in members:
            image = read_image(fits_files[j])
            dx = (j - center) * drift[0] + sat_x[j] - sat_x[center]
            dy = (j - center) * drift[1] + sat_y[j] - sat_y[center]
            shifted = ndimage.shift(image, (-dy, -dx), order=1, mode="constant", cval=np.nan, prefilter=False)
            ok = np.isfinite(shifted)
            if total is None:
                total = np.zeros_like(image)
                weight = np.zeros_like(image)
            total[ok] += shifted[ok]
            weight += ok
        n = len(members)
        stack = np.where(weight > 0, total * (n / np.maximum(weight, 1)), 0.0).astype(np.float32)
        with fits.open(fits_files[center], memmap=False) as hdul:
            header = hdul[0].header.copy()
        header["EXPTIME"] = (float(header["EXPTIME"]) * n, "Summed exposure of the stack [s]")
        header["NSTACK"] = (n, "Number of frames summed")
        header["STACKCTR"] = (int(center), "Zero-based centre frame index")
        header["DRIFTX"] = (float(drift[0]), "Star drift x [pixel/frame]")
        header["DRIFTY"] = (float(drift[1]), "Star drift y [pixel/frame]")
        path = output_directory / f"stack_{center + 1:05d}.fits"
        fits.writeto(path, stack, header, overwrite=True, output_verify="silentfix")
        return {"center_index": int(center), "source_path": path, "n_frames": n}

    with ThreadPoolExecutor(max_workers=min(16, len(centers))) as executor:
        records = list(tqdm(executor.map(build_one, centers), total=len(centers), desc="Building star-aligned stacks"))
    return records


def plate_solve_stacks(stack_records, solved_directory, work_directory, max_workers, center_hint):
    '''
    Extract a clean star list from every stack and plate-solve the lists
    (parallel). Records gain solved_path (stack + WCS), wcs, header, rdls_path.
    '''
    solved_directory = recreate_directory(solved_directory)
    work_directory = recreate_directory(work_directory)

    def solve_one(record):
        with fits.open(record["source_path"], memmap=False) as hdul:
            image = np.asarray(hdul[0].data, dtype=np.float32)
            header = hdul[0].header.copy()
        frame_work = work_directory / record["source_path"].stem
        frame_work.mkdir(parents=True, exist_ok=True)
        stars = extract_star_list(image)
        xyls_path = frame_work / f"{record['source_path'].stem}.xyls"
        stars.write(xyls_path, format="fits", overwrite=True)
        if len(stars) < 4:
            return {**record, "solved": False, "error": f"only {len(stars)} usable stars", "n_stars": len(stars)}
        wcs_header, rdls_path, error = solve_star_list(
            xyls_path, image.shape[1], image.shape[0], frame_work, center_hint,
        )
        if wcs_header is None:
            return {**record, "solved": False, "error": error, "n_stars": len(stars)}
        merged = merge_wcs_header(header, wcs_header)
        solved_path = solved_directory / record["source_path"].name
        fits.writeto(solved_path, image, merged, overwrite=True, output_verify="silentfix")
        return {**record, "solved": True, "error": None, "n_stars": len(stars),
                "solved_path": solved_path, "header": merged, "wcs": WCS(merged), "rdls_path": rdls_path}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        output = list(tqdm(executor.map(solve_one, stack_records), total=len(stack_records), desc="Plate solving stacks"))
    for r in output:
        if not r["solved"]:
            print(f"Plate solve failed for {r['source_path'].name}: {r['error']}")
            r.setdefault("wcs", None)
            r.setdefault("header", None)
    solved = [r for r in output if r["solved"]]
    print(f"Solved {len(solved)} / {len(output)} stacks")
    if solved:
        first = solved[0]["wcs"]
        scale = np.sqrt(np.abs(np.linalg.det(first.pixel_scale_matrix))) * 3600.0
        sky = first.pixel_to_world(first.pixel_shape[0] / 2.0, first.pixel_shape[1] / 2.0)
        print(f"  pixel scale {scale:.4f} arcsec/px, field centre RA {sky.ra.deg:.4f} Dec {sky.dec.deg:.4f} "
              f"({solved[0]['n_stars']} stars used)")
    return output


def derive_frame_wcs(fits_files, stack_records, sat_x, sat_y, drift):
    '''
    WCS for every frame from the solved stacks.

    Between two consecutive solved stacks (centre frames c_k < c_k+1) the
    field-star shift is measured directly: the sky position of the centre
    pixel of stack k is located in stack k+1, and the tracking jitter measured
    on the target, s(j), is removed. This gives a local drift v_k. A frame j is
    attached to the nearest solved stack, whose WCS is shifted in CRPIX by
    (j - c_k) v_local + s(j) - s(c_k). Sky points are never evaluated far
    outside a stack, so the distortion terms are not extrapolated.
    '''
    solved = sorted([r for r in stack_records if r["solved"]], key=lambda r: r["center_index"])
    if not solved:
        raise RuntimeError("No stack was plate-solved; cannot derive frame WCS.")
    with fits.open(fits_files[0], memmap=False) as hdul:
        ny, nx = hdul[0].data.shape
    cx, cy = (nx - 1) / 2.0, (ny - 1) / 2.0
    centers = np.array([r["center_index"] for r in solved], dtype=float)

    # Local drift between consecutive solved stacks (jitter removed).
    local_v = []
    for k in range(len(solved) - 1):
        a, b = solved[k], solved[k + 1]
        sky = a["wcs"].pixel_to_world(cx, cy)
        qb = np.asarray(b["wcs"].world_to_pixel(sky), dtype=float)
        ca, cb = int(a["center_index"]), int(b["center_index"])
        shift = (qb - np.array([cx, cy])) - (np.array([sat_x[cb], sat_y[cb]]) - np.array([sat_x[ca], sat_y[ca]]))
        local_v.append(shift / (cb - ca))
    if local_v:
        local_v = np.array(local_v)
        v = np.median(local_v, axis=0)
        spread = np.std(local_v, axis=0)
        print(f"Local star drift between stacks: median v = ({v[0]:.4f}, {v[1]:.4f}) px/frame, "
              f"std over intervals = ({spread[0]:.4f}, {spread[1]:.4f}) px/frame; "
              f"blob estimate was ({drift[0]:.3f}, {drift[1]:.3f})")
    else:
        local_v = np.array([np.asarray(drift, dtype=float)])
        v = local_v[0]
        print("Only one stack solved; using the blob drift estimate for all frames.")

    def drift_for(j):
        if len(local_v) == 1:
            return local_v[0]
        k = int(np.clip(np.searchsorted(centers, j, side="right") - 1, 0, len(local_v) - 1))
        return local_v[k]

    records = []
    for j, path in enumerate(fits_files):
        k = int(np.argmin(np.abs(centers - j)))
        base = solved[k]
        c = int(base["center_index"])
        delta = (j - c) * drift_for(j if j < c else max(j - 1, c)) + np.array([sat_x[j] - sat_x[c], sat_y[j] - sat_y[c]])
        header = base["header"].copy()
        header["CRPIX1"] = float(header["CRPIX1"]) + float(delta[0])
        header["CRPIX2"] = float(header["CRPIX2"]) + float(delta[1])
        header["EXPTIME"] = base["header"]["EXPTIME"] / base["n_frames"]
        header["WCSSTACK"] = (c, "Stack centre frame whose WCS was shifted")
        records.append({"source_path": path, "solved_path": path, "solved": True,
                        "wcs": WCS(header), "header": header})
    model = {"v_px_per_frame": [float(v[0]), float(v[1])],
             "local_v_px_per_frame": [[float(a), float(b)] for a, b in local_v],
             "method": "CRPIX shift from the nearest solved stack with local drift and target jitter",
             "n_stacks_solved": len(solved), "stack_centers": [int(c) for c in centers]}
    return records, model

### Star subtraction

In [ ]:
def choose_reference_index(index, frame_count, interval):
    """Choose a separated frame while retaining every science frame."""
    if interval < 1:
        raise ValueError("star_reference_interval must be at least 1.")
    if frame_count <= interval:
        raise ValueError(
            "The number of frames must exceed star_reference_interval."
        )
    later = index + interval
    if later < frame_count:
        return later
    return index - interval


def shift_image(image, delta_x, delta_y):
    """Shift an image in detector x/y coordinates without integer wrapping."""
    finite = np.isfinite(image)
    filled = np.where(finite, image, 0.0)
    shifted_data = ndimage.shift(
        filled,
        shift=(delta_y, delta_x),
        order=star_shift_order,
        mode=star_shift_mode,
        cval=0.0,
        prefilter=star_shift_order > 1,
    )
    shifted_weight = ndimage.shift(
        finite.astype(float),
        shift=(delta_y, delta_x),
        order=1,
        mode="constant",
        cval=0.0,
        prefilter=False,
    )
    shifted_data[shifted_weight < 0.99] = star_shift_cval
    return shifted_data

def subtract_field_stars_single_frame(
    index,
    records,
    star_x,
    star_y,
    output_directory,
):
    """
    Subtract a star-aligned reference frame from one frame.

    Parameters
    ----------
    index : int
        Index of the current science frame.

    records : list[dict]
        Astrometry records for all frames.

    star_x, star_y : ndarray
        Interpolated detector coordinates of the common
        celestial position in every frame.

    output_directory : pathlib.Path
        Directory in which the star-subtracted FITS file
        will be saved.

    Returns
    -------
    output_record : dict
        Updated record containing the star-subtracted FITS
        path and reference-frame information.
    """

    frame_count = len(records)

    record = records[index]

    reference_index = choose_reference_index(
        index,
        frame_count,
        int(star_reference_interval),
    )

    reference_record = records[
        reference_index
    ]

    # Read the current science frame.
    with fits.open(
        record["source_path"],
        memmap=False,
    ) as hdul:
        data = np.asarray(
            hdul[0].data,
            dtype=float,
        )

        header = hdul[0].header.copy()

    # Read the reference frame.
    with fits.open(
        reference_record["source_path"],
        memmap=False,
    ) as hdul:
        reference_data = np.asarray(
            hdul[0].data,
            dtype=float,
        )

    # Shift the reference image so that its field stars
    # coincide with those in the current frame.
    delta_x = (
        star_x[index]
        - star_x[reference_index]
    )

    delta_y = (
        star_y[index]
        - star_y[reference_index]
    )

    aligned_reference = shift_image(
        reference_data,
        delta_x,
        delta_y,
    )

    # Subtract the aligned field-star image.
    star_subtracted = (
        data
        - aligned_reference
    )

    output_path = (
        Path(output_directory)
        / record["source_path"].name
    )

    header["STARSUB"] = (
        True,
        "Field-star reference subtraction applied",
    )

    header["REFINDEX"] = (
        int(reference_index),
        "Zero-based reference frame index",
    )

    header["REFDX"] = (
        float(delta_x),
        "Reference shift in detector x [pixel]",
    )

    header["REFDY"] = (
        float(delta_y),
        "Reference shift in detector y [pixel]",
    )

    fits.writeto(
        output_path,
        star_subtracted.astype(np.float32),
        header,
        overwrite=True,
        output_verify="silentfix",
    )

    return {
        **record,
        "photometry_path": output_path,
        "reference_index": int(
            reference_index
        ),
        "reference_shift_x": float(
            delta_x
        ),
        "reference_shift_y": float(
            delta_y
        ),
    }

def subtract_field_stars(
    records,
    output_directory,
    parallel=True,
    max_workers=4,
):
    """
    Subtract star-aligned reference frames from all
    satellite-tracking frames.

    Parameters
    ----------
    records : list[dict]
        Astrometry records for all frames.

    output_directory : str or pathlib.Path
        Output directory for star-subtracted FITS files.

    parallel : bool, optional
        If True, process frames concurrently.

    max_workers : int, optional
        Maximum number of simultaneous subtraction tasks.

    Returns
    -------
    output_records : list[dict]
        Records in the original chronological frame order.
    """

    output_directory = recreate_directory(
        output_directory
    )

    star_x, star_y = (
        interpolate_star_positions(
            records
        )
    )

    frame_count = len(records)

    if frame_count == 0:
        return []

    max_workers = int(
        max_workers
    )

    if max_workers < 1:
        raise ValueError(
            "star_subtraction_max_workers must be at least 1."
        )

    # --------------------------------------------------------
    # Sequential processing
    # --------------------------------------------------------

    if (
        not parallel
        or max_workers == 1
    ):
        output_records = []

        for index in tqdm(
            range(frame_count),
            desc="Subtracting field stars",
        ):
            output_record = (
                subtract_field_stars_single_frame(
                    index=index,
                    records=records,
                    star_x=star_x,
                    star_y=star_y,
                    output_directory=(
                        output_directory
                    ),
                )
            )

            output_records.append(
                output_record
            )

        return output_records

    # --------------------------------------------------------
    # Parallel processing
    # --------------------------------------------------------

    output_records = [
        None
    ] * frame_count

    with ThreadPoolExecutor(
        max_workers=max_workers
    ) as executor:

        future_to_index = {
            executor.submit(
                subtract_field_stars_single_frame,
                index,
                records,
                star_x,
                star_y,
                output_directory,
            ): index
            for index in range(
                frame_count
            )
        }

        for future in tqdm(
            as_completed(
                future_to_index
            ),
            total=frame_count,
            desc="Subtracting field stars",
        ):
            index = future_to_index[
                future
            ]

            try:
                output_records[index] = (
                    future.result()
                )

            except Exception as error:
                raise RuntimeError(
                    "Field-star subtraction failed for "
                    f"frame {index}: "
                    f"{records[index]['source_path'].name}"
                ) from error

    return output_records

### Photometry

In [ ]:
def detect_brightest_positive_source(
    image,
    detection_sigma=6.0,
    cutout_half_size=25,
):
    """Detect the brightest positive point-like residual in a 2-D image."""
    image = np.asarray(image, dtype=float)
    finite = np.isfinite(image)
    if not np.any(finite):
        raise ValueError("The detection image contains no finite pixels.")
    _, median, std = sigma_clipped_stats(image[finite], sigma=3.0, maxiters=5)
    if not np.isfinite(std) or std <= 0:
        raise ValueError("Could not estimate detection-image noise.")

    cleaned = np.where(finite, image, median)
    threshold = median + detection_sigma * std
    labels, number = ndimage.label(cleaned > threshold)
    if number == 0:
        raise RuntimeError("No target candidate was detected.")

    best_label = None
    best_flux = -np.inf
    for label_number in range(1, number + 1):
        region = labels == label_number
        flux = np.sum(np.clip(cleaned[region] - median, 0, None))
        if flux > best_flux:
            best_flux = flux
            best_label = label_number

    yy, xx = np.nonzero(labels == best_label)
    rough_x = float(np.mean(xx))
    rough_y = float(np.mean(yy))
    ny, nx = image.shape
    x0 = max(0, int(round(rough_x)) - cutout_half_size)
    x1 = min(nx, int(round(rough_x)) + cutout_half_size + 1)
    y0 = max(0, int(round(rough_y)) - cutout_half_size)
    y1 = min(ny, int(round(rough_y)) + cutout_half_size + 1)
    cutout = cleaned[y0:y1, x0:x1]
    _, local_median, _ = sigma_clipped_stats(cutout, sigma=3.0, maxiters=5)
    source = np.clip(cutout - local_median, 0, None)
    x_local, y_local = centroid_2dg(source)
    return float(x0 + x_local), float(y0 + y_local)


def build_target_detection_image(records, sample_count=30):
    """Median-combine positive star-subtracted samples for robust target detection."""
    if not records:
        raise ValueError("No star-subtracted records are available.")
    sample_indices = np.unique(
        np.linspace(0, len(records) - 1, min(sample_count, len(records))).astype(int)
    )
    stack = []
    for index in sample_indices:
        with fits.open(records[index]["photometry_path"], memmap=False) as hdul:
            stack.append(np.asarray(hdul[0].data, dtype=float))
    return np.nanmedian(np.stack(stack), axis=0)


def refine_centroid(image, x, y, half_size=8):
    """Refine a source centroid locally with a 2-D Gaussian centroid."""
    image = np.asarray(image, dtype=float)
    ny, nx = image.shape
    xc = int(round(float(x)))
    yc = int(round(float(y)))
    x0 = max(0, xc - half_size)
    x1 = min(nx, xc + half_size + 1)
    y0 = max(0, yc - half_size)
    y1 = min(ny, yc + half_size + 1)
    cutout = image[y0:y1, x0:x1]
    finite = np.isfinite(cutout)
    if np.count_nonzero(finite) < 9:
        raise ValueError("Too few finite pixels for centroid refinement.")
    _, median, _ = sigma_clipped_stats(cutout[finite], sigma=3.0, maxiters=5)
    source = np.where(finite, np.clip(cutout - median, 0, None), 0.0)
    x_local, y_local = centroid_2dg(source)
    return float(x0 + x_local), float(y0 + y_local)


def measure_object(image, x_object, y_object, aperture_radius, annulus_r_in, annulus_r_out):
    """Perform circular-aperture photometry with a sigma-clipped annulus."""
    image = np.asarray(image, dtype=float)
    position = (float(x_object), float(y_object))
    aperture = CircularAperture(position, r=float(aperture_radius))
    annulus = CircularAnnulus(
        position,
        r_in=float(annulus_r_in),
        r_out=float(annulus_r_out),
    )
    phot_table = aperture_photometry(image, aperture, method="exact")
    aperture_sum = float(phot_table["aperture_sum"][0])
    annulus_stats = ApertureStats(
        image,
        annulus,
        sigma_clip=SigmaClip(sigma=3.0, maxiters=5),
    )
    background_median = float(annulus_stats.median)
    aperture_area = float(aperture.area)
    background_sum = background_median * aperture_area
    object_flux = aperture_sum - background_sum
    return {
        "aperture_sum": aperture_sum,
        "background_median": background_median,
        "background_sum": background_sum,
        "aperture_area": aperture_area,
        "object_flux": object_flux,
    }


def read_photometric_zeropoint(magzero_path):
    """
    Prefer photometric_zeropoint_elec. If unavailable, use
    photometric_zeropoint_exp without detector-gain correction.
    """
    with Path(magzero_path).open("r", encoding="utf-8") as file:
        data = json.load(file)
    source = data.get("phot_params", data)

    zp_elec = source.get("photometric_zeropoint_elec", source.get("ZP_elec"))
    try:
        zp_elec = float(zp_elec)
    except (TypeError, ValueError):
        zp_elec = np.nan
    if np.isfinite(zp_elec):
        return data, zp_elec, "elec"

    zp_exp = source.get("photometric_zeropoint_exp", source.get("ZP_exp"))
    try:
        zp_exp = float(zp_exp)
    except (TypeError, ValueError):
        zp_exp = np.nan
    if np.isfinite(zp_exp):
        return data, zp_exp, "exp"

    raise KeyError(
        "Neither a finite photometric_zeropoint_elec nor a finite "
        "photometric_zeropoint_exp is available."
    )


def calculate_apparent_magnitude(object_flux, exptime, header, zeropoint, mode):
    """Calculate apparent magnitude with the selected zeropoint convention."""
    if not np.isfinite(object_flux) or object_flux <= 0:
        return np.nan, np.nan, np.nan

    instrumental_mag = -2.5 * np.log10(object_flux)
    if mode == "elec":
        if "EGAINSAV" not in header:
            raise KeyError(
                "photometric_zeropoint_elec is being used, but EGAINSAV is "
                "missing from the science FITS header."
            )
        egainsav = float(header["EGAINSAV"])
        if not np.isfinite(egainsav) or egainsav <= 0:
            raise ValueError(f"Invalid EGAINSAV: {egainsav}")
        apparent_mag = (
            instrumental_mag
            + 2.5 * np.log10(exptime)
            - 2.5 * np.log10(egainsav)
            + zeropoint
        )
    elif mode == "exp":
        egainsav = np.nan
        apparent_mag = instrumental_mag + 2.5 * np.log10(exptime) + zeropoint
    else:
        raise ValueError(f"Unknown zeropoint mode: {mode}")
    return instrumental_mag, apparent_mag, egainsav

### Output

In [ ]:
def json_value(value):
    """Convert NumPy/Astropy scalar values to JSON-compatible values."""
    if np.ma.is_masked(value):
        return None
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    return value


def export_kupt_lightcurve_json(
    table,
    output_path,
    fits_header,
    magzero_data,
    zeropoint_mode,
    target_info=None,
):
    """Write a compact KUPT-lightcurveJSON version 2 product."""
    valid_rows = [row for row in table if str(row["time_start_utc"]).strip()]
    if not valid_rows:
        raise ValueError("No valid timestamp is available for JSON output.")

    first = valid_rows[0]
    last = valid_rows[-1]
    start = Time(str(first["time_start_utc"]), scale="utc")
    end = Time(str(last["time_start_utc"]), scale="utc") + TimeDelta(
        float(last["exptime_seconds"]), format="sec"
    )
    phot_source = magzero_data.get("phot_params", magzero_data)

    data_rows = []
    for row in table:
        data_rows.append({
            "id": int(row["frame_index"]),
            "time_start_utc": str(row["time_start_utc"]) or None,
            "time_mid_utc": str(row["time_mid_utc"]) or None,
            "elapsed_seconds": json_value(row["elapsed_seconds"]),
            "x_centroid": json_value(row["x_centroid"]),
            "y_centroid": json_value(row["y_centroid"]),
            "aperture_sum": json_value(row["aperture_sum_adu"]),
            "bkg_med_pix": json_value(row["background_median_adu_per_pixel"]),
            "object_flux": json_value(row["object_flux_adu"]),
            "mag_in": json_value(row["instrumental_mag"]),
            "mag_app": json_value(row["apparent_mag"]),
            "status": str(row["status"]),
            "comment": {
                "general": [
                    f"Source FITS file: {row['filename']}",
                    f"Star reference frame index: {row['star_reference_index']}",
                    f"Zeropoint mode: {zeropoint_mode}",
                ]
            },
        })

    output = {
        "header": {
            "object": {
                "objectName": object_name,
                "intlDES": intl_des,
                "noradID": None if norad_id is None else str(norad_id),
            },
            "startDateTime": start.isot,
            "endDateTime": end.isot,
            "length": len(table),
            "captureSettings": {
                "Xnaxis": json_value(fits_header.get("NAXIS1")),
                "Ynaxis": json_value(fits_header.get("NAXIS2")),
                "Xbinning": json_value(fits_header.get("XBINNING", 1)),
                "Ybinning": json_value(fits_header.get("YBINNING", 1)),
                "exposure": json_value(fits_header.get("EXPTIME")),
                "gain": json_value(fits_header.get("GAIN")),
                "egain": json_value(fits_header.get("EGAIN")),
                "egainsav": json_value(fits_header.get("EGAINSAV")),
                "offset": json_value(fits_header.get("OFFSET")),
                "camName": json_value(fits_header.get("INSTRUME")),
                "camID": json_value(fits_header.get("CAMID")),
                "swcreate": json_value(fits_header.get("SWCREATE")),
            },
            "photParams": {
                "pixel_area_arcsec2": json_value(phot_source.get("pixel_area_arcsec2")),
                "aperture_area_arcsec2": json_value(phot_source.get("aperture_area_arcsec2")),
                "ZP": json_value(phot_source.get("photometric_zeropoint")),
                "ZP_exp": json_value(phot_source.get("photometric_zeropoint_exp")),
                "ZP_elec": json_value(phot_source.get("photometric_zeropoint_elec")),
                "ZP_scatter": json_value(phot_source.get("zeropoint_scatter")),
                "selected_mode": zeropoint_mode,
            },
            "observer": {
                "observerName": observer_name,
                "observatory": observatory,
            },
            "editor": {
                "appName": editor_app_name,
                "version": editor_version,
            },
            "measureSettings": {
                "measureFieldShape": "CIRCLE",
                "measureFieldSize": {
                    "aperture_radius_pixel": float(aperture_radius),
                    "annulus_r_in_pixel": float(annulus_r_in),
                    "annulus_r_out_pixel": float(annulus_r_out),
                },
                "targetPosition": sanitize_json(target_info or {}),
            },
        },
        "data": data_rows,
        "createdBy": {
            "appName": creator_app_name,
            "version": creator_version,
            "built": int(datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")),
        },
        "dataType": "KUPT-lightcurveJSON",
        "version": 2,
        "comment": {
            "general": [
                "GEO satellite-tracking photometry with automatic target detection, "
                "integrated field-star zeropoint and WCS-based field-star subtraction."
            ],
            "TLE": [],
        },
    }

    # Remove absent optional capture settings while retaining mandatory dimensions.
    output["header"]["captureSettings"] = {
        key: value
        for key, value in output["header"]["captureSettings"].items()
        if value is not None
    }
    output["header"]["photParams"] = {
        key: value
        for key, value in output["header"]["photParams"].items()
        if value is not None
    }

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(output, file, ensure_ascii=False, indent=2, allow_nan=False)
    return output

### Target detection and review

In [ ]:
def estimate_noise(image, floor=0.25):
    """
    Robust background noise. 8-bit frames whose sky is clipped at zero give a
    sigma-clipped std of 0; fall back to the MAD of the non-zero pixels and
    finally to the quantisation floor.
    """
    values = np.asarray(image, dtype=float)
    values = values[np.isfinite(values)]
    _, _, std = sigma_clipped_stats(values, sigma=3.0, maxiters=5)
    if not np.isfinite(std) or std <= 0:
        nonzero = values[values != 0]
        std = 1.4826 * np.median(np.abs(nonzero - np.median(nonzero))) if nonzero.size > 100 else 0.0
    return float(max(std, floor))


def choose_sample_indices(frame_count, sample_count):
    """Evenly spaced frame indices."""
    if sample_count is None or int(sample_count) >= frame_count:
        return np.arange(frame_count, dtype=int)
    return np.unique(
        np.linspace(0, frame_count - 1, int(sample_count)).round().astype(int)
    )


hot_pixel_mask = None   # set by stage 1 / loaded by stage 2


def clean_hot(image, mask=None):
    """Replace masked pixels by the 3x3 median."""
    mask = hot_pixel_mask if mask is None else mask
    if mask is None:
        return image
    filled = ndimage.median_filter(np.where(mask, 0.0, image), 3)
    return np.where(mask, filled, image).astype(np.float32)


def read_image(path, clean=True):
    with fits.open(path, memmap=False) as hdul:
        image = np.asarray(hdul[0].data, dtype=np.float32)
    return clean_hot(image) if clean else image


def build_raw_detection_image(fits_files, sample_count):
    """Median of frames sampled across the capture (stars drift out, target stays)."""
    indices = choose_sample_indices(len(fits_files), sample_count)
    stack = np.stack(
        [read_image(fits_files[i]) for i in tqdm(indices, desc="Loading detection samples")]
    )
    return np.nanmedian(stack, axis=0), indices


def measure_local_snr(image, x, y, radius, r_in, r_out, noise_floor=0.0):
    """Background-subtracted aperture signal-to-noise and peak at one position."""
    ny, nx = image.shape
    if not (r_out <= x < nx - r_out and r_out <= y < ny - r_out):
        return np.nan, np.nan
    aperture = CircularAperture((x, y), r=radius)
    annulus = CircularAnnulus((x, y), r_in=r_in, r_out=r_out)
    stats = ApertureStats(image, annulus, sigma_clip=SigmaClip(sigma=3.0, maxiters=5))
    background = float(stats.median)
    noise = float(stats.std)
    if not np.isfinite(noise) or noise <= 0:
        noise = float(noise_floor)
    total = float(aperture_photometry(image, aperture, method="exact")["aperture_sum"][0])
    net = total - background * aperture.area
    if not np.isfinite(noise) or noise <= 0:
        return np.nan, np.nan
    snr = net / (noise * np.sqrt(aperture.area))
    x0, x1 = int(x - radius), int(np.ceil(x + radius)) + 1
    y0, y1 = int(y - radius), int(np.ceil(y + radius)) + 1
    peak = float(np.nanmax(image[y0:y1, x0:x1]))
    return float(snr), peak


def find_target_candidates(
    detection_image,
    sample_files,
    *,
    sigma,
    smoothing_sigma,
    min_area,
    max_elongation,
    snr_radius,
    n_candidates,
    saturation_level,
):
    """
    Find compact sources in the detection image and rank them by how
    persistently they appear at the same position in the individual frames.
    """
    image = np.asarray(detection_image, dtype=float)
    finite = np.isfinite(image)
    _, median, _ = sigma_clipped_stats(image[finite], sigma=3.0, maxiters=5)
    std = estimate_noise(image[finite])

    cleaned = np.where(finite, image, median)
    smoothed = ndimage.gaussian_filter(cleaned, smoothing_sigma)
    labels, number = ndimage.label(smoothed > median + sigma * std)
    if number == 0:
        raise RuntimeError("No source above the detection threshold. Lower detection_sigma.")

    candidates = []
    for label_number, slices in enumerate(ndimage.find_objects(labels), start=1):
        region = labels[slices] == label_number
        area = int(np.count_nonzero(region))
        if area < min_area:
            continue
        sub = cleaned[slices] - median
        weights = np.clip(sub, 0, None) * region
        total = float(weights.sum())
        if total <= 0:
            continue
        yy, xx = np.indices(sub.shape)
        xc = float((weights * xx).sum() / total)
        yc = float((weights * yy).sum() / total)
        dx, dy = xx - xc, yy - yc
        cxx = float((weights * dx * dx).sum() / total)
        cyy = float((weights * dy * dy).sum() / total)
        cxy = float((weights * dx * dy).sum() / total)
        eigenvalues = np.linalg.eigvalsh([[cxx, cxy], [cxy, cyy]])
        elongation = (
            float(np.sqrt(eigenvalues[1] / eigenvalues[0])) if eigenvalues[0] > 0 else np.inf
        )
        peak = float(cleaned[slices][region].max() - median)
        peak_fraction = peak / total if total > 0 else 1.0
        if peak_fraction > detection_max_peak_fraction or not np.isfinite(elongation):
            continue   # single hot pixel or cosmic ray, not a PSF
        candidates.append({
            "x": xc + slices[1].start,
            "y": yc + slices[0].start,
            "area_pixel": area,
            "flux_detection_image": total,
            "peak_detection_image": peak,
            "peak_fraction": float(peak_fraction),
            "elongation": elongation,
        })

    if not candidates:
        raise RuntimeError("Every detected source is smaller than detection_min_area.")

    # Keep the brightest sources for the (slower) persistence check.
    candidates.sort(key=lambda c: -c["flux_detection_image"])
    candidates = candidates[: max(n_candidates * 4, 10)]

    snr_lists = [[] for _ in candidates]
    peak_lists = [[] for _ in candidates]
    for path in tqdm(sample_files, desc="Checking persistence in individual frames"):
        frame = read_image(path)
        frame_std = estimate_noise(frame[::8, ::8])
        for k, c in enumerate(candidates):
            snr, peak = measure_local_snr(
                frame, c["x"], c["y"], snr_radius, snr_radius + 3.0, snr_radius + 8.0,
                noise_floor=frame_std,
            )
            snr_lists[k].append(snr)
            peak_lists[k].append(peak)

    for k, c in enumerate(candidates):
        snr = np.asarray(snr_lists[k], dtype=float)
        peak = np.asarray(peak_lists[k], dtype=float)
        c["median_snr"] = float(np.nanmedian(snr))
        c["detected_fraction"] = float(np.nanmean(snr > 3.0))
        c["saturated_fraction"] = float(np.nanmean(peak >= saturation_level))
        c["elongated"] = bool(c["elongation"] > max_elongation)
        score = 100.0 * c["detected_fraction"] + min(c["median_snr"], 50.0)
        if c["elongated"]:
            score -= 100.0
        c["score"] = float(score)

    candidates.sort(key=lambda c: -c["score"])
    for rank, c in enumerate(candidates, start=1):
        c["rank"] = rank
    return candidates[:n_candidates]


def cutout(image, x, y, half_size):
    ny, nx = image.shape
    x0 = max(0, int(round(x)) - half_size)
    x1 = min(nx, int(round(x)) + half_size + 1)
    y0 = max(0, int(round(y)) - half_size)
    y1 = min(ny, int(round(y)) + half_size + 1)
    return image[y0:y1, x0:x1], x0, y0


def create_review_figure(
    detection_image,
    candidates,
    sample_files,
    sample_indices,
    output_path,
    *,
    cutout_half_size=30,
    n_frame_cutouts=8,
    title="",
):
    """Full field with ranked candidates, candidate cutouts, and the best candidate in individual frames."""
    n_cols = max(len(candidates), n_frame_cutouts, 1)
    figure = plt.figure(figsize=(2.2 * n_cols, 2.2 * n_cols * 0.75 + 5.5))
    grid = figure.add_gridspec(3, n_cols, height_ratios=[n_cols * 0.75, 1.0, 1.0], hspace=0.35)

    axis = figure.add_subplot(grid[0, :])
    norm = ImageNormalize(detection_image, interval=ZScaleInterval(), stretch=AsinhStretch())
    axis.imshow(detection_image, cmap="gray", norm=norm, origin="lower")
    for c in candidates:
        color = "lime" if c["rank"] == 1 else "orange"
        axis.add_patch(plt.Circle((c["x"], c["y"]), 40, fill=False, color=color, linewidth=1.2))
        axis.text(c["x"] + 45, c["y"] + 45, f"#{c['rank']}", color=color, fontsize=10, weight="bold")
    axis.set_title(f"{title}\nDetection image (median of {len(sample_indices)} frames); #1 = adopted candidate")
    axis.set_xlabel("x [pixel]")
    axis.set_ylabel("y [pixel]")

    for k, c in enumerate(candidates):
        axis = figure.add_subplot(grid[1, k])
        sub, x0, y0 = cutout(detection_image, c["x"], c["y"], cutout_half_size)
        axis.imshow(sub, cmap="gray", origin="lower",
                    norm=ImageNormalize(sub, interval=ZScaleInterval(), stretch=AsinhStretch()))
        axis.plot(c["x"] - x0, c["y"] - y0, "+", color="lime" if c["rank"] == 1 else "orange", markersize=12)
        axis.set_title(
            f"#{c['rank']} ({c['x']:.1f}, {c['y']:.1f})\n"
            f"SNR {c['median_snr']:.1f}  seen {100 * c['detected_fraction']:.0f}%"
            + ("  SAT" if c["saturated_fraction"] > 0 else "")
            + ("  elong" if c["elongated"] else ""),
            fontsize=8,
        )
        axis.set_xticks([])
        axis.set_yticks([])

    best = candidates[0]
    frame_pick = np.unique(
        np.linspace(0, len(sample_files) - 1, min(n_frame_cutouts, len(sample_files))).round().astype(int)
    )
    for k, j in enumerate(frame_pick):
        axis = figure.add_subplot(grid[2, k])
        frame = read_image(sample_files[j])
        sub, x0, y0 = cutout(frame, best["x"], best["y"], cutout_half_size)
        axis.imshow(sub, cmap="gray", origin="lower",
                    norm=ImageNormalize(sub, interval=ZScaleInterval(), stretch=AsinhStretch()))
        axis.plot(best["x"] - x0, best["y"] - y0, "+", color="lime", markersize=12)
        axis.set_title(f"#1 in frame {int(sample_indices[j]) + 1}", fontsize=8)
        axis.set_xticks([])
        axis.set_yticks([])

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_path, dpi=130, bbox_inches="tight")
    plt.close(figure)
    return output_path


def locate_target(image, x0, y0, half_size, max_shift):
    """
    Re-centre the target in one frame: brightest smoothed blob inside a window
    around the detected position, then a flux-weighted centroid around it.
    Falls back to (x0, y0) when nothing usable is found or the offset is too large.
    """
    sub, xs, ys = cutout(image, x0, y0, half_size)
    finite = np.isfinite(sub)
    if np.count_nonzero(finite) < 9:
        return float(x0), float(y0), "fixed_position_fallback"
    source = np.where(finite, sub - np.nanmedian(sub[finite]), 0.0)
    source[source < 0] = 0.0
    smoothed = ndimage.gaussian_filter(source, 1.5)
    if not np.any(smoothed > 0):
        return float(x0), float(y0), "fixed_position_fallback"
    py, px = np.unravel_index(int(np.argmax(smoothed)), smoothed.shape)
    w = 6
    y0w, y1w = max(0, py - w), min(source.shape[0], py + w + 1)
    x0w, x1w = max(0, px - w), min(source.shape[1], px + w + 1)
    window = source[y0w:y1w, x0w:x1w]
    total = float(window.sum())
    if total <= 0:
        return float(x0), float(y0), "fixed_position_fallback"
    yy, xx = np.indices(window.shape)
    xc = float((window * xx).sum() / total) + x0w + xs
    yc = float((window * yy).sum() / total) + y0w + ys
    if np.hypot(xc - x0, yc - y0) > max_shift:
        return float(x0), float(y0), "fixed_position_fallback"
    return xc, yc, "local_centroid"


def print_candidates(candidates):
    print(f"{'rank':>4s} {'x':>9s} {'y':>9s} {'SNR':>7s} {'seen':>6s} {'sat':>5s} {'elong':>6s} {'area':>5s}")
    for c in candidates:
        print(
            f"{c['rank']:4d} {c['x']:9.2f} {c['y']:9.2f} {c['median_snr']:7.1f} "
            f"{100 * c['detected_fraction']:5.0f}% {100 * c['saturated_fraction']:4.0f}% "
            f"{c['elongation']:6.2f} {c['area_pixel']:5d}"
        )

### Zeropoint from field stars (magzero integrated)

In [ ]:
def table_to_records(table):
    return [{name: json_value(row[name]) for name in table.colnames} for row in table]


def header_to_observation(header):
    observation = {}
    for card in header.cards:
        keyword = str(card.keyword).strip()
        if keyword.upper() in {"COMMENT", "HISTORY", ""}:
            continue
        observation[keyword] = {
            "value": json_value(card.value),
            "comment": str(card.comment) if card.comment is not None else "",
        }
    return observation


def find_rdls_file(work_directory):
    """RDLS (catalogue stars in the field) written by solve-field in a frame work directory."""
    files = sorted(Path(work_directory).glob("*.rdls"))
    return files[0] if files else None


def measure_frame_zeropoint(solved_path, rdls_path):
    """Zeropoint of one plate-solved frame from catalogue stars (magzero.ipynb method)."""
    rdls = Table.read(rdls_path)
    with fits.open(solved_path, memmap=False) as hdul:
        header = hdul[0].header.copy()
        image = np.asarray(hdul[0].data, dtype=float)
    wcs = WCS(header).celestial
    ny, nx = image.shape
    mask = ~np.isfinite(image)

    sky = SkyCoord(
        ra=np.asarray(rdls["RA"], dtype=float) * u.deg,
        dec=np.asarray(rdls["DEC"], dtype=float) * u.deg,
        frame="icrs",
    )
    x_init, y_init = wcs.world_to_pixel(sky)
    x_init = np.asarray(x_init, dtype=float)
    y_init = np.asarray(y_init, dtype=float)
    half_box = zp_box_size // 2
    margin = max(zp_annulus_r_out + 1, half_box)
    inside = (
        np.isfinite(x_init) & np.isfinite(y_init)
        & (x_init >= margin) & (x_init < nx - margin)
        & (y_init >= margin) & (y_init < ny - margin)
    )
    stars = rdls[inside].copy()
    x_init, y_init = x_init[inside], y_init[inside]
    if len(stars) < 3:
        raise RuntimeError("Fewer than three catalogue stars fall inside the frame.")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        x_ref, y_ref = centroid_sources(
            image, xpos=x_init, ypos=y_init, box_size=zp_box_size,
            mask=mask, centroid_func=centroid_2dg,
        )
    shift = np.hypot(x_ref - x_init, y_ref - y_init)
    good = np.isfinite(x_ref) & np.isfinite(y_ref) & (shift <= zp_max_shift)
    x_fin = np.where(good, x_ref, x_init)
    y_fin = np.where(good, y_ref, y_init)
    positions = np.column_stack([x_fin, y_fin])

    aperture = CircularAperture(positions, r=zp_aperture_radius)
    annulus = CircularAnnulus(positions, r_in=zp_annulus_r_in, r_out=zp_annulus_r_out)
    stats = ApertureStats(image, annulus, mask=mask, sigma_clip=SigmaClip(sigma=3.0, maxiters=5))
    background = np.asarray(stats.median, dtype=float)
    raw = np.asarray(aperture_photometry(image, aperture, mask=mask, method="exact")["aperture_sum"], dtype=float)
    net = raw - background * aperture.area

    pixel_scales = proj_plane_pixel_scales(wcs)
    pixel_area_arcsec2 = float(pixel_scales[0] * pixel_scales[1] * 3600.0 ** 2)
    aperture_area_arcsec2 = float(aperture.area * pixel_area_arcsec2)

    catalog_mag = np.asarray(stars["mag"], dtype=float)
    positive = np.isfinite(net) & (net > 0) & np.isfinite(catalog_mag)
    inst = np.full(len(net), np.nan)
    inst[positive] = -2.5 * np.log10(net[positive])
    zp_each = catalog_mag - inst
    selection = positive & (catalog_mag > reference_mag_min) & (catalog_mag < reference_mag_max)
    if np.count_nonzero(selection) < 3:
        raise RuntimeError("Fewer than three reference stars in the magnitude range.")
    initial = np.nanmedian(zp_each[selection])
    selection &= np.abs(zp_each - initial) < 0.5
    zeropoint = float(np.nanmedian(zp_each[selection]))
    scatter = float(np.nanstd(zp_each[selection]))
    exptime = float(header["EXPTIME"])
    zeropoint_exp = zeropoint - 2.5 * np.log10(exptime)
    zeropoint_elec = (
        zeropoint_exp + 2.5 * np.log10(float(header["EGAINSAV"])) if "EGAINSAV" in header else None
    )

    stars["x_wcs"], stars["y_wcs"] = x_init, y_init
    stars["x_refined"], stars["y_refined"] = x_fin, y_fin
    stars["centroid_shift"], stars["centroid_success"] = shift, good
    stars["aperture_sum_raw"], stars["local_background_per_pixel"] = raw, background
    stars["net_flux"], stars["instrumental_mag"] = net, inst
    stars["zeropoint"], stars["used_for_zeropoint"] = zp_each, selection

    return {
        "solved_path": str(solved_path),
        "rdls_path": str(rdls_path),
        "date_obs": str(header.get("DATE-OBS", "")),
        "exptime": exptime,
        "pixel_area_arcsec2": pixel_area_arcsec2,
        "aperture_area_arcsec2": aperture_area_arcsec2,
        "photometric_zeropoint": zeropoint,
        "photometric_zeropoint_exp": float(zeropoint_exp),
        "photometric_zeropoint_elec": None if zeropoint_elec is None else float(zeropoint_elec),
        "zeropoint_scatter": scatter,
        "n_reference_stars": int(np.count_nonzero(selection)),
        "header": header,
        "stars": stars,
    }


def measure_integrated_zeropoint(astrometry_records, work_directory, output_directory, frame_count):
    """Median zeropoint over several plate-solved frames; writes a MAGZERO JSON."""
    solved = [r for r in astrometry_records if r["solved"] and r["header"] is not None]
    if not solved:
        raise RuntimeError("No plate-solved frame is available for the zeropoint.")
    indices = choose_sample_indices(len(solved), frame_count)

    frame_results = []
    for i in tqdm(indices, desc="Measuring zeropoint"):
        record = solved[i]
        rdls_path = record.get("rdls_path") or find_rdls_file(Path(work_directory) / record["source_path"].stem)
        if rdls_path is None:
            print(f"  No RDLS file for {record['source_path'].name}; skipped.")
            continue
        try:
            frame_results.append(measure_frame_zeropoint(record["solved_path"], rdls_path))
        except Exception as error:
            print(f"  Zeropoint failed for {record['source_path'].name}: {error}")
    if not frame_results:
        raise RuntimeError("The zeropoint could not be measured on any frame.")

    def combine(key):
        values = np.asarray([r[key] for r in frame_results if r[key] is not None], dtype=float)
        return (float(np.nanmedian(values)), float(np.nanstd(values))) if values.size else (None, None)

    zp, zp_frame_std = combine("photometric_zeropoint")
    zp_exp, _ = combine("photometric_zeropoint_exp")
    zp_elec, _ = combine("photometric_zeropoint_elec")
    scatter = float(np.nanmedian([r["zeropoint_scatter"] for r in frame_results]))
    first = frame_results[0]

    magzero_data = {
        "metadata": {
            "creator": "Kiyoaki Okudaira",
            "created_at": datetime.now(ZoneInfo("Asia/Tokyo")).isoformat(timespec="seconds"),
            "software": {
                "name": "SatPhotometry MAGZERO pipeline (GEO SatTrack integrated)",
                "platform": platform.platform(),
            },
        },
        "phot_params": {
            "pixel_area_arcsec2": first["pixel_area_arcsec2"],
            "aperture_area_arcsec2": first["aperture_area_arcsec2"],
            "photometric_zeropoint": zp,
            "photometric_zeropoint_exp": zp_exp,
            "photometric_zeropoint_elec": zp_elec,
            "zeropoint_scatter": scatter,
            "zeropoint_frame_to_frame_std": zp_frame_std,
            "n_frames": len(frame_results),
            "aperture_radius_pixel": float(zp_aperture_radius),
            "annulus_r_in_pixel": float(zp_annulus_r_in),
            "annulus_r_out_pixel": float(zp_annulus_r_out),
            "reference_mag_range": [reference_mag_min, reference_mag_max],
        },
        "observation": header_to_observation(first["header"]),
        "frames": [
            {k: v for k, v in r.items() if k not in {"header", "stars"}} for r in frame_results
        ],
        "ref_stars": {
            "columns": first["stars"].colnames,
            "data": table_to_records(first["stars"]),
        },
    }

    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)
    stamp = str(first["header"]["DATE-OBS"])[0:19].replace(":", "")
    gain = first["header"].get("GAIN")
    gain_text = f"_GAIN{int(gain)}" if gain is not None else ""
    json_path = output_directory / f"magzero_{stamp}{gain_text}_integrated.json"
    with json_path.open("w", encoding="utf-8") as file:
        json.dump(magzero_data, file, ensure_ascii=False, indent=2)

    print(f"Zeropoint from {len(frame_results)} frame(s): ZP = {zp:.4f}, ZP_exp = {zp_exp:.4f}, "
          f"scatter = {scatter:.4f} mag, frame-to-frame std = {zp_frame_std:.4f} mag")
    print(f"Saved MAGZERO JSON: {json_path}")
    return magzero_data, json_path

### Stage 1: prepare frames and detect the target

In [ ]:
def capture_base_name():
    safe_object = re.sub(r"[^A-Za-z0-9._-]+", "_", object_name).strip("_")
    return f"{safe_object}_{PATH_capture.stem}"


def stage1_state_path():
    return PATH_work / f"stage1_state_{capture_base_name()}.json"


def determine_calibration_scale(master, fits_files):
    """Scale for a master frame whose bit depth differs from the light frames."""
    if calibration_scale != "auto":
        return float(calibration_scale)
    if master is None:
        return 1.0
    with fits.open(fits_files[0], memmap=False) as hdul:
        light_max = float(np.nanmax(hdul[0].data))
    master_max = float(np.nanmax(master))
    if light_max <= 255.0 and master_max > 255.0:
        return 1.0 / 256.0
    return 1.0


def stage1_prepare_and_detect():
    """SER -> FITS, calibration, detection image, candidates, review figure."""
    PATH_output.mkdir(parents=True, exist_ok=True)
    PATH_work.mkdir(parents=True, exist_ok=True)
    raw_fits_directory = PATH_work / "fits_raw"

    if input_format.upper() == "SER":
        fits_files = convert_ser_to_fits(
            PATH_capture,
            raw_fits_directory,
            start_frame=ser_start_frame,
            end_frame=ser_end_frame,
            exposure_seconds=ser_exposure_seconds,
            gain=ser_gain,
            egain=ser_egain,
            egainsav=ser_egainsav,
            offset=ser_offset,
            x_binning=ser_x_binning,
            y_binning=ser_y_binning,
            camera_name=ser_camera_name,
            camera_id=ser_camera_id,
            ccd_temperature=ser_ccd_temperature,
            set_temperature=ser_set_temperature,
            observer=observer_name,
            observatory=observatory,
            object_name=object_name,
            intl_des=intl_des,
            norad_id=norad_id,
        )
    elif input_format.upper() == "FITS":
        source_files = get_fits_files(PATH_capture)
        recreate_directory(raw_fits_directory)
        fits_files = []
        for source in source_files:
            destination = raw_fits_directory / source.name
            shutil.copy2(source, destination)
            fits_files.append(destination)
    else:
        raise ValueError("input_format must be either 'SER' or 'FITS'.")

    # Dark/flat are NOT applied here: astrometry (stage 2) must run on the raw
    # frames. Calibrated copies are made in stage 2 for the photometry.
    global hot_pixel_mask
    hot_pixel_mask = None
    if remove_hot_pixels and PATH_dark is not None:
        master_dark = create_master_dark(PATH_dark)
        master_dark = master_dark * determine_calibration_scale(master_dark, fits_files)
        hot_pixel_mask = build_dark_hot_pixel_mask(master_dark, hot_pixel_sigma)
        print(f"Warm pixels from the master dark: {int(hot_pixel_mask.sum())} "
              f"({100.0 * hot_pixel_mask.mean():.2f}% of the frame)")

    detection_image, sample_indices = build_raw_detection_image(fits_files, detection_sample_count)
    detection_path = PATH_work / f"target_detection_image_{capture_base_name()}.fits"
    fits.writeto(detection_path, detection_image, overwrite=True)

    sample_files = [fits_files[i] for i in sample_indices]
    candidates = find_target_candidates(
        detection_image,
        sample_files,
        sigma=detection_sigma,
        smoothing_sigma=detection_smoothing_sigma,
        min_area=detection_min_area,
        max_elongation=detection_max_elongation,
        snr_radius=detection_snr_radius,
        n_candidates=n_candidates,
        saturation_level=saturation_level,
    )
    for c in candidates:
        try:
            c["x"], c["y"] = refine_centroid(detection_image, c["x"], c["y"], half_size=centroid_half_size)
        except Exception:
            pass

    hot_mask_path = None
    if remove_hot_pixels:
        hot_mask = build_hot_pixel_mask(detection_image, candidates, hot_pixel_sigma, hot_pixel_exclude_radius)
        if hot_pixel_mask is not None:
            hot_mask |= hot_pixel_mask
        hot_mask_path = PATH_work / f"hot_pixel_mask_{capture_base_name()}.fits"
        fits.writeto(hot_mask_path, hot_mask.astype(np.uint8), overwrite=True)
        print(f"Hot pixels: {int(hot_mask.sum())} ({100.0 * hot_mask.mean():.2f}% of the frame); "
              "they are replaced in memory when frames are read (raw files stay untouched)")
        hot_pixel_mask = hot_mask

    review_path = create_review_figure(
        detection_image,
        candidates,
        sample_files,
        sample_indices,
        PATH_output / "review" / f"{capture_base_name()}_target_review.png",
        title=f"{object_name} ({intl_des})  {PATH_capture.name}",
    )

    state = {
        "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "capture": str(PATH_capture),
        "fits_files": [str(p) for p in fits_files],
        "detection_image": str(detection_path),
        "detection_sample_indices": [int(i) for i in sample_indices],
        "candidates": candidates,
        "review_figure": str(review_path),
        "hot_pixel_mask": None if hot_mask_path is None else str(hot_mask_path),
        "calibration": {"dark": str(PATH_dark), "flat": str(PATH_flat), "flat_dark": str(PATH_flat_dark)},
    }
    with stage1_state_path().open("w", encoding="utf-8") as file:
        json.dump(state, file, ensure_ascii=False, indent=2)

    print()
    print(f"Frames prepared     : {len(fits_files)}")
    print(f"Detection image     : {detection_path}")
    print(f"Review figure       : {review_path}")
    print(f"Stage-1 state       : {stage1_state_path()}")
    print()
    print_candidates(candidates)
    return state


def load_stage1_state():
    global hot_pixel_mask
    path = stage1_state_path()
    if not path.is_file():
        raise FileNotFoundError(f"Stage-1 state not found: {path}. Run stage 1 first.")
    with path.open("r", encoding="utf-8") as file:
        state = json.load(file)
    hot_pixel_mask = None
    if state.get("hot_pixel_mask"):
        hot_pixel_mask = fits.getdata(state["hot_pixel_mask"]).astype(bool)
    return state


def select_target(state):
    """Target position from the manual override or the chosen candidate."""
    if target_x is not None and target_y is not None:
        return float(target_x), float(target_y), {"method": "manual", "x": float(target_x), "y": float(target_y)}
    if target_x is not None or target_y is not None:
        raise ValueError("target_x and target_y must both be set or both be None.")
    candidates = state["candidates"]
    if not 0 <= target_candidate_index < len(candidates):
        raise IndexError("target_candidate_index is outside the candidate list.")
    c = candidates[target_candidate_index]
    return float(c["x"]), float(c["y"]), {
        "method": "automatic detection (median of sampled frames, persistence ranked)",
        "candidate_rank": int(c["rank"]),
        "x": float(c["x"]),
        "y": float(c["y"]),
        "median_snr": c["median_snr"],
        "detected_fraction": c["detected_fraction"],
        "saturated_fraction": c["saturated_fraction"],
        "approved_by_operator": bool(require_confirmation and target_approved),
        "candidates": candidates,
    }

### Stage 2: plate solve, zeropoint, star subtraction, photometry

In [ ]:
def stage2_measure(state):
    """Run the measurement on the frames prepared in stage 1."""
    fits_files = [Path(p) for p in state["fits_files"]]
    solved_directory = PATH_work / "fits_solved"
    astrometry_work_directory = PATH_work / "astrometry_work"
    star_subtracted_directory = PATH_work / "fits_star_subtracted"
    product_directory = PATH_output / "lightcurve"
    product_directory.mkdir(parents=True, exist_ok=True)

    x_fixed, y_fixed, target_info = select_target(state)
    print(f"Target position: x = {x_fixed:.3f}, y = {y_fixed:.3f} ({target_info['method']})")

    sat_x, sat_y, sat_methods = measure_target_positions(fits_files, x_fixed, y_fixed)

    # ---- astrometry on the RAW frames
    if astrometry_mode == "frames":
        astrometry_records = plate_solve_files_parallel(
            fits_files=fits_files,
            solved_directory=solved_directory,
            work_directory=astrometry_work_directory,
            max_workers=astrometry_max_workers,
        )
        stack_records = None
        drift = None
    elif astrometry_mode == "stacks":
        drift = star_drift_px_per_frame
        if drift is None:
            drift = estimate_star_drift(fits_files, sat_x, sat_y)
        stack_records = build_star_aligned_stacks(
            fits_files, sat_x, sat_y, drift, astrometry_stack_frames, astrometry_stack_count,
            PATH_work / "astrometry_stacks",
        )
        stack_records = plate_solve_stacks(
            stack_records, solved_directory, astrometry_work_directory, astrometry_max_workers, read_center_hint(),
        )
        astrometry_records, drift_model = derive_frame_wcs(fits_files, stack_records, sat_x, sat_y, drift)
        target_info["star_drift_model"] = drift_model
    else:
        raise ValueError("astrometry_mode must be 'stacks' or 'frames'.")
    n_solved = sum(1 for r in astrometry_records if r["solved"])
    print(f"WCS available for {n_solved} / {len(astrometry_records)} frames")

    # ---- dark/flat calibration (after astrometry)
    master_flat_dark = create_master_dark(PATH_flat_dark)
    master_dark = create_master_dark(PATH_dark)
    master_flat = create_master_flat(PATH_flat, master_flat_dark)
    dark_scale = determine_calibration_scale(master_dark, fits_files)
    if master_dark is not None and dark_scale != 1.0:
        print(f"Master dark scaled by {dark_scale:.6f} to match the light-frame bit depth.")
        master_dark = master_dark * dark_scale
    calibrated_files = calibrate_frames(fits_files, PATH_work / "fits_cal", dark=master_dark, flat=master_flat)
    calibrated_by_name = {p.name: p for p in calibrated_files}
    for record in astrometry_records:
        record["source_path"] = calibrated_by_name[Path(record["source_path"]).name]
        record["solved_path"] = record["source_path"]

    # ---- zeropoint frames: calibrated stacks carrying the WCS of the raw solved stacks
    if stack_records is not None:
        calibrated_stacks = build_star_aligned_stacks(
            calibrated_files, sat_x, sat_y, drift, astrometry_stack_frames, astrometry_stack_count,
            PATH_work / "astrometry_stacks_cal",
        )
        zeropoint_records = []
        for raw, cal in zip(stack_records, calibrated_stacks):
            if not raw["solved"]:
                continue
            with fits.open(cal["source_path"], memmap=False) as hdul:
                data = np.asarray(hdul[0].data, dtype=np.float32)
                header = hdul[0].header.copy()
            merged = merge_wcs_header(header, raw["header"])
            solved_path = Path(cal["source_path"]).with_name(Path(cal["source_path"]).stem + "_wcs.fits")
            fits.writeto(solved_path, data, merged, overwrite=True, output_verify="silentfix")
            zeropoint_records.append({**cal, "solved": True, "solved_path": solved_path,
                                      "header": merged, "wcs": WCS(merged), "rdls_path": raw["rdls_path"]})
        zeropoint_work_directory = astrometry_work_directory
    else:
        zeropoint_records, zeropoint_work_directory = astrometry_records, astrometry_work_directory

    if magzero_source == "integrated":
        _, magzero_path = measure_integrated_zeropoint(
            zeropoint_records, zeropoint_work_directory, PATH_output / "magzero", zeropoint_frame_count,
        )
    elif magzero_source == "file":
        magzero_path = PATH_magzero
    else:
        raise ValueError("magzero_source must be 'integrated' or 'file'.")
    magzero_data, zeropoint, zeropoint_mode = read_photometric_zeropoint(magzero_path)
    print(f"Using zeropoint mode: {zeropoint_mode} ({magzero_path.name})")

    if subtract_stars:
        photometry_records = subtract_field_stars(
            records=astrometry_records,
            output_directory=star_subtracted_directory,
            parallel=star_subtraction_parallel,
            max_workers=star_subtraction_max_workers,
        )
    else:
        photometry_records = [
            {**record, "photometry_path": record["source_path"], "reference_index": -1,
             "reference_shift_x": np.nan, "reference_shift_y": np.nan}
            for record in astrometry_records
        ]

    results = []
    first_time_mid = None
    first_header = None
    previous_x, previous_y = x_fixed, y_fixed
    for frame_index, record in enumerate(tqdm(photometry_records, desc="Measuring GEO target")):
        path = record["photometry_path"]
        status = "Success"
        try:
            with fits.open(path, memmap=False) as hdul:
                image = np.asarray(hdul[0].data, dtype=float)
                header = hdul[0].header.copy()
            if first_header is None:
                first_header = header.copy()
            time_start, time_mid, exptime = get_observation_time(header)
            if first_time_mid is None:
                first_time_mid = time_mid
            elapsed = float((time_mid - first_time_mid).to_value("sec"))

            if fix_target_position:
                x_object, y_object = x_fixed, y_fixed
                position_method = "fixed_position"
            else:
                x_object, y_object = float(sat_x[frame_index]), float(sat_y[frame_index])
                position_method = sat_methods[frame_index]

            phot = measure_object(image, x_object, y_object, aperture_radius, annulus_r_in, annulus_r_out)
            sub, _, _ = cutout(image, x_object, y_object, int(np.ceil(aperture_radius)))
            peak = float(np.nanmax(sub)) if sub.size else np.nan
            instrumental_mag, apparent_mag, egainsav = calculate_apparent_magnitude(
                phot["object_flux"], exptime, header, zeropoint, zeropoint_mode,
            )
            if not np.isfinite(apparent_mag):
                status = "Non-positive flux"
            elif np.isfinite(peak) and peak >= saturation_level:
                status = "Saturated"

            results.append({
                "frame_index": frame_index,
                "filename": path.name,
                "time_start_utc": time_start.isot,
                "time_mid_utc": time_mid.isot,
                "elapsed_seconds": elapsed,
                "exptime_seconds": exptime,
                "egainsav_electron_per_adu": egainsav,
                "x_centroid": x_object,
                "y_centroid": y_object,
                "position_method": position_method,
                "star_reference_index": record["reference_index"],
                "star_reference_shift_x": record["reference_shift_x"],
                "star_reference_shift_y": record["reference_shift_y"],
                "aperture_radius_pixel": aperture_radius,
                "annulus_r_in_pixel": annulus_r_in,
                "annulus_r_out_pixel": annulus_r_out,
                "aperture_area_pixel2": phot["aperture_area"],
                "aperture_sum_adu": phot["aperture_sum"],
                "background_median_adu_per_pixel": phot["background_median"],
                "background_sum_adu": phot["background_sum"],
                "object_flux_adu": phot["object_flux"],
                "peak_adu": peak,
                "instrumental_mag": instrumental_mag,
                "apparent_mag": apparent_mag,
                "zeropoint_mode": zeropoint_mode,
                "status": status,
            })
        except Exception as error:
            print(f"Frame {frame_index} ({path.name}) failed: {error}")
            results.append({
                "frame_index": frame_index,
                "filename": path.name,
                "time_start_utc": "",
                "time_mid_utc": "",
                "elapsed_seconds": np.nan,
                "exptime_seconds": np.nan,
                "egainsav_electron_per_adu": np.nan,
                "x_centroid": np.nan,
                "y_centroid": np.nan,
                "position_method": "",
                "star_reference_index": record["reference_index"],
                "star_reference_shift_x": record["reference_shift_x"],
                "star_reference_shift_y": record["reference_shift_y"],
                "aperture_radius_pixel": aperture_radius,
                "annulus_r_in_pixel": annulus_r_in,
                "annulus_r_out_pixel": annulus_r_out,
                "aperture_area_pixel2": np.nan,
                "aperture_sum_adu": np.nan,
                "background_median_adu_per_pixel": np.nan,
                "background_sum_adu": np.nan,
                "object_flux_adu": np.nan,
                "peak_adu": np.nan,
                "instrumental_mag": np.nan,
                "apparent_mag": np.nan,
                "zeropoint_mode": zeropoint_mode,
                "status": f"Error: {error}",
            })

    table = Table(rows=results)
    valid = [row for row in results if str(row["time_start_utc"]).strip()]
    if not valid:
        raise RuntimeError("No frame was measured successfully.")
    n_saturated = sum(1 for row in results if row["status"] == "Saturated")
    if n_saturated:
        print(f"Warning: {n_saturated} frame(s) reached the saturation level ({saturation_level}).")

    start = Time(valid[0]["time_start_utc"], scale="utc")
    end = Time(valid[-1]["time_start_utc"], scale="utc") + TimeDelta(valid[-1]["exptime_seconds"], format="sec")
    safe_object = re.sub(r"[^A-Za-z0-9._-]+", "_", object_name).strip("_")
    safe_intldes = re.sub(r"[^A-Za-z0-9._-]+", "_", str(intl_des)).strip("_")
    base_name = (
        f"lightcurve_{safe_object}_{safe_intldes}_"
        f"{start.datetime.strftime('%Y-%m-%d_%H%M%S')}_"
        f"{end.datetime.strftime('%H%M%S')}"
    )
    csv_path = product_directory / f"{base_name}.csv"
    json_path = product_directory / f"{base_name}.json"
    table.write(csv_path, format="ascii.csv", overwrite=True)
    json_output = export_kupt_lightcurve_json(
        table, json_path, first_header, magzero_data, zeropoint_mode, target_info=target_info,
    )
    print(f"Saved CSV:  {csv_path}")
    print(f"Saved JSON: {json_path}")
    return table, csv_path, json_path, json_output

### Run stage 1

In [ ]:
if run_stage in ("all", "1"):
    stage1_state = stage1_prepare_and_detect()
else:
    stage1_state = load_stage1_state()
    print(f"Loaded stage-1 state from {stage1_state_path()}")
    print_candidates(stage1_state["candidates"])

display(Image(filename=stage1_state["review_figure"]))

### Review the detected target
Check the figure above (also saved under `output/review/`): candidate **#1** is adopted.
The bottom row shows #1 in individual frames; the object should stay on the cross while stars drift.

- Correct → set `target_approved = True` below and run the remaining cells.
- Another candidate is the object → set `target_candidate_index` in the parameters and run from stage 1 again (or set `target_x` / `target_y` manually).
- Unattended runs: `require_confirmation = False` (or `SATPHOT_REQUIRE_CONFIRMATION=0`), or `SATPHOT_TARGET_APPROVED=1` after reviewing the figure.

In [ ]:
# target_approved = True

### Run stage 2

In [ ]:
if run_stage not in ("all", "2"):
    print("Stage 2 skipped (run_stage).")
elif require_confirmation and not target_approved:
    print("Stage 2 is waiting for the operator: review the figure, set target_approved = True, and run this cell again.")
else:
    lightcurve_table, output_csv_path, output_json_path, kupt_json_data = stage2_measure(stage1_state)

### Result

In [ ]:
if "lightcurve_table" in globals():
    elapsed_seconds = np.asarray(lightcurve_table["elapsed_seconds"], dtype=float)
    apparent_mag = np.asarray(lightcurve_table["apparent_mag"], dtype=float)
    valid = np.isfinite(elapsed_seconds) & np.isfinite(apparent_mag)

    plt.figure(figsize=(9, 5))
    plt.plot(elapsed_seconds[valid], apparent_mag[valid], marker="o", markersize=3, linewidth=0.8)
    plt.xlabel("Elapsed time from first frame [s]")
    plt.ylabel("Apparent magnitude [mag]")
    plt.title(f"{object_name} ({intl_des})")
    plt.gca().invert_yaxis()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PATH_output / "lightcurve" / (Path(output_csv_path).stem + ".png"), dpi=150)
    plt.show()